# Portable core training template (adapted after the experiments)
This template removes private module imports and supports the architectures used in the audit.
It preserves the training functions. It has not been GPU retrained during preflight.
Attach the repository as a Kaggle dataset containing code/skin_disease, set SKIN31_ROOT and TRAINING_SEED,
and follow notebooks/README.md. The archived training_kaggle.ipynb remains unmodified.


# Install + imports

In [ ]:
!pip -q install timm==1.0.26


In [ ]:
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image, ImageEnhance
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.optim import Optimizer
from torch.utils.data import DataLoader, ConcatDataset, WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.transforms import functional as TF
import torch.nn.functional as F

import timm
from tqdm.auto import tqdm


import gc
import json
from pathlib import Path

In [ ]:
import torch, torchvision, timm, PIL, sklearn, numpy, sys
print(sys.version)
print("torch", torch.__version__, "cuda", torch.version.cuda, "cudnn", torch.backends.cudnn.version())
print("torchvision", torchvision.__version__, "timm", timm.__version__, "Pillow", PIL.__version__)
print("sklearn", sklearn.__version__, "numpy", numpy.__version__)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")


In [ ]:
import sys
from pathlib import Path
# Locate the attached release package; no private architecture modules required.
code_roots = [Path.cwd() / "code"] + list(Path("/kaggle/input").glob("**/code"))
code_root = next((p for p in code_roots if (p / "skin_disease").is_dir()), None)
if code_root is None:
    raise FileNotFoundError("Attach code/skin_disease from this repository as a Kaggle dataset.")
sys.path.insert(0, str(code_root))
from skin_disease.models import efficientnet_b2_original, efficientnet_b2_dual_level as efficientnet_b2_dual_scale


# Global config

In [ ]:
seed = int(os.environ.get("TRAINING_SEED", "42"))

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset_name = "skin31"
# ================================================================
# DATA PROTOCOL
# ================================================================

# DATA_MODE = "dev"
DATA_MODE = "final"

# Chỉ có ý nghĩa khi DATA_MODE == "final"
# True  = log test mỗi epoch, nhưng KHÔNG dùng test để chọn model
# False = scientific strict: chỉ test 1 lần sau epoch cuối
FINAL_TEST_EACH_EPOCH = True
skin31_root = Path(os.environ["SKIN31_ROOT"])
skin31_train_dir = skin31_root / "train"
skin31_test_dir = skin31_root / "test"

ham_root = Path("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000")
ham_metadata_path = ham_root / "HAM10000_metadata.csv"
ham_part1_dir = ham_root / "HAM10000_images_part_1"
ham_part2_dir = ham_root / "HAM10000_images_part_2"

work_dir = Path("/kaggle/working") / f"seed{seed}"
work_dir.mkdir(parents=True, exist_ok=True)
history_dir = work_dir / "histories"
history_dir.mkdir(parents=True, exist_ok=True)
class_names_path = work_dir / f"class_names_{dataset_name}.json"

img_size = 224
batch_size = 32
epochs = 15
lr = 5e-4
weight_decay = 1e-6
num_workers = 2

In [ ]:
def reset_experiment_seed(seed_value=42):
    random.seed(seed_value)
    np.random.seed(seed_value)

    torch.manual_seed(seed_value)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)

    # DataLoader generator đã tồn tại trong code cũ
    g.manual_seed(seed_value)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Dataset overview

In [ ]:
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split

HAM_DX_TO_NAME = {
    "akiec": "Actinic keratoses and intraepithelial carcinoma",
    "bcc": "Basal cell carcinoma",
    "bkl": "Benign keratosis-like lesions",
    "df": "Dermatofibroma",
    "mel": "Melanoma",
    "nv": "Melanocytic nevi",
    "vasc": "Vascular lesions"
}

def build_ham10000_dataframe(metadata_path, part1_dir, part2_dir):
    df = pd.read_csv(metadata_path).copy()

    def resolve_path(image_id):
        p1 = part1_dir / f"{image_id}.jpg"
        p2 = part2_dir / f"{image_id}.jpg"
        if p1.exists():
            return str(p1)
        if p2.exists():
            return str(p2)
        return None

    df["path"] = df["image_id"].map(resolve_path)
    df = df[df["path"].notna()].reset_index(drop=True)

    dx_order = sorted(df["dx"].unique().tolist())
    class_names = [HAM_DX_TO_NAME.get(x, x) for x in dx_order]
    dx_to_idx = {dx: i for i, dx in enumerate(dx_order)}

    df["target"] = df["dx"].map(dx_to_idx)
    df["class_name"] = df["dx"].map(lambda x: HAM_DX_TO_NAME.get(x, x))

    return df, dx_order, class_names, dx_to_idx

class HAM10000Dataset(Dataset):
    def __init__(self, df, class_names, transform=None, exclude_paths=None):
        self.df = df.copy().reset_index(drop=True)
        exclude_paths = {str(Path(p)) for p in (exclude_paths or [])}

        if len(exclude_paths) > 0:
            self.df = self.df[~self.df["path"].isin(exclude_paths)].reset_index(drop=True)

        self.transform = transform
        self.classes = class_names
        self.samples = list(zip(self.df["path"].tolist(), self.df["target"].tolist()))
        self.imgs = self.samples
        self.targets = self.df["target"].tolist()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        target = int(row["target"])

        if self.transform is not None:
            image = self.transform(image)

        return image, target


def build_ham10000_split(test_size=0.2, seed=42):
    ham_df, ham_dx_order, ham_class_names, ham_dx_to_idx = build_ham10000_dataframe(
        metadata_path=ham_metadata_path,
        part1_dir=ham_part1_dir,
        part2_dir=ham_part2_dir
    )

    lesion_df = (
        ham_df.groupby("lesion_id", as_index=False)["target"]
        .first()
        .copy()
    )

    train_lesions, test_lesions = train_test_split(
        lesion_df["lesion_id"],
        test_size=test_size,
        random_state=seed,
        stratify=lesion_df["target"]
    )

    train_df = ham_df[ham_df["lesion_id"].isin(train_lesions)].reset_index(drop=True)
    test_df = ham_df[ham_df["lesion_id"].isin(test_lesions)].reset_index(drop=True)

    train_df = train_df.copy()
    test_df = test_df.copy()

    return train_df, test_df, ham_class_names

In [ ]:
def summarize_ham_split(train_df, test_df, class_names):
    train_counts = train_df["target"].value_counts().sort_index()
    test_counts = test_df["target"].value_counts().sort_index()

    summary_df = pd.DataFrame({
        "class_name": class_names,
        "train_count": [int(train_counts.get(i, 0)) for i in range(len(class_names))],
        "test_count": [int(test_counts.get(i, 0)) for i in range(len(class_names))]
    })

    print("Train images:", len(train_df))
    print("Test images:", len(test_df))
    print("Train lesions:", train_df["lesion_id"].nunique())
    print("Test lesions:", test_df["lesion_id"].nunique())

    display(summary_df)

In [ ]:
if dataset_name == "ham10000":
    ham_train_df, ham_test_df, class_names = build_ham10000_split(test_size=0.2, seed=42)
    summarize_ham_split(ham_train_df, ham_test_df, class_names)

In [ ]:
if dataset_name == "skin31":
    base_train = datasets.ImageFolder(skin31_train_dir)
    base_test = datasets.ImageFolder(skin31_test_dir)
    if base_train.class_to_idx != base_test.class_to_idx:
        raise ValueError("Train/test class names and output order differ")

    class_names = base_train.classes
    num_classes = len(class_names)

    train_counts = pd.Series(base_train.targets).value_counts().sort_index()
    test_counts = pd.Series(base_test.targets).value_counts().sort_index()

elif dataset_name == "ham10000":
    ham_train_df, ham_test_df, class_names = build_ham10000_split(test_size=0.2, seed=seed)

    base_train = HAM10000Dataset(ham_train_df, class_names=class_names, transform=None)
    base_test = HAM10000Dataset(ham_test_df, class_names=class_names, transform=None)

    num_classes = len(class_names)

    train_counts = pd.Series(base_train.targets).value_counts().sort_index()
    test_counts = pd.Series(base_test.targets).value_counts().sort_index()

else:
    raise ValueError(f"Unsupported dataset_name: {dataset_name}")

df_counts = pd.DataFrame({
    "class_name": class_names,
    "train_count": train_counts.values,
    "test_count": test_counts.values
})

class_meta = {
    "dataset_name": dataset_name,
    "num_classes": num_classes,
    "class_names": class_names
}

with open(class_names_path, "w", encoding="utf-8") as f:
    json.dump(class_meta, f, ensure_ascii=False, indent=2)

print("Dataset:", dataset_name)
print("Num classes:", num_classes)
print("Train samples:", len(base_train))
print("Test samples:", len(base_test))

display(df_counts)

# Custom transforms

In [ ]:
class FixedCenterZoom:
    def __init__(self, scale=1.2):
        self.scale = scale

    def __call__(self, img):
        w, h = img.size
        nw, nh = int(w * self.scale), int(h * self.scale)
        img = img.resize((nw, nh), Image.BILINEAR)
        left = (nw - w) // 2
        top = (nh - h) // 2
        return img.crop((left, top, left + w, top + h))


class FixedRotate:
    def __init__(self, angle=20):
        self.angle = angle

    def __call__(self, img):
        return TF.rotate(img, self.angle, interpolation=transforms.InterpolationMode.BILINEAR, fill=0)


class FixedBrightness:
    def __init__(self, factor=1.2):
        self.factor = factor

    def __call__(self, img):
        return ImageEnhance.Brightness(img).enhance(self.factor)


class FixedShear:
    def __init__(self, shear=12):
        self.shear = shear

    def __call__(self, img):
        return TF.affine(
            img,
            angle=0,
            translate=[0, 0],
            scale=1.0,
            shear=[self.shear, 0],
            interpolation=transforms.InterpolationMode.BILINEAR,
            fill=0
        )

sample_path, sample_label = base_train.samples[0]
sample_img = Image.open(sample_path).convert("RGB")

preview = {
    "Original": sample_img,
    "Center Zoom": FixedCenterZoom(1.2)(sample_img.copy()),
    "Rotation": FixedRotate(20)(sample_img.copy()),
    "Brightness": FixedBrightness(1.2)(sample_img.copy()),
    "Shear": FixedShear(12)(sample_img.copy()),
    "Vertical Flip": TF.vflip(sample_img.copy()),
    "Horizontal Flip": TF.hflip(sample_img.copy())
}

fig, axes = plt.subplots(1, 7, figsize=(24, 4))

for ax, (name, img) in zip(axes, preview.items()):
    ax.imshow(img)
    ax.set_title(name)
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Sample class:", class_names[sample_label])

# Dataset filtering helper

In [ ]:
class FilteredImageFolder(datasets.ImageFolder):
    def __init__(self, root, transform=None, exclude_paths=None):
        super().__init__(root=root, transform=transform)
        exclude_paths = {str(Path(p)) for p in (exclude_paths or [])}

        filtered_samples = []
        filtered_targets = []

        for path, target in self.samples:
            if str(Path(path)) not in exclude_paths:
                filtered_samples.append((path, target))
                filtered_targets.append(target)

        self.samples = filtered_samples
        self.imgs = filtered_samples
        self.targets = filtered_targets

# Transform definitions

In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

to_tensor_norm = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])
orig_tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

zoom_tf = transforms.Compose([
    FixedCenterZoom(1.2),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

rot_tf = transforms.Compose([
    FixedRotate(20),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

bright_tf = transforms.Compose([
    FixedBrightness(1.2),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

shear_tf = transforms.Compose([
    FixedShear(12),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

vflip_tf = transforms.Compose([
    transforms.Lambda(lambda x: TF.vflip(x)),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

hflip_tf = transforms.Compose([
    transforms.Lambda(lambda x: TF.hflip(x)),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

# Training utilities

In [ ]:
def rand_bbox(size, lam):
    H = size[2]
    W = size[3]

    cut_rat = np.sqrt(1.0 - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    x1 = np.clip(cx - cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y2 = np.clip(cy + cut_h // 2, 0, H)

    return x1, y1, x2, y2


def apply_cutmix(images, labels, alpha=1.0):
    if alpha <= 0:
        return images, labels, labels, 1.0

    lam = np.random.beta(alpha, alpha)
    rand_index = torch.randperm(images.size(0), device=images.device)

    labels_a = labels
    labels_b = labels[rand_index]

    x1, y1, x2, y2 = rand_bbox(images.size(), lam)
    images = images.clone()
    images[:, :, y1:y2, x1:x2] = images[rand_index, :, y1:y2, x1:x2]

    lam = 1.0 - ((x2 - x1) * (y2 - y1) / (images.size(2) * images.size(3)))

    return images, labels_a, labels_b, lam

In [ ]:
class SAM(Optimizer):
    def __init__(self, params, base_optimizer_cls, rho=0.05, adaptive=False, **kwargs):
        if rho < 0.0:
            raise ValueError(f"Invalid rho, should be non-negative: {rho}")

        defaults = dict(rho=rho, adaptive=adaptive, **kwargs)
        super().__init__(params, defaults)

        self.base_optimizer = base_optimizer_cls(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
        self.defaults.update(self.base_optimizer.defaults)
        self.rho = rho
        self.adaptive = adaptive

    @torch.no_grad()
    def _grad_norm(self):
        device = self.param_groups[0]["params"][0].device
        norms = []

        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                if group.get("adaptive", self.adaptive):
                    norms.append((torch.abs(p) * p.grad).norm(p=2).to(device))
                else:
                    norms.append(p.grad.norm(p=2).to(device))

        if len(norms) == 0:
            return torch.tensor(0.0, device=device)

        return torch.norm(torch.stack(norms), p=2)

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()

        if grad_norm.item() == 0.0:
            if zero_grad:
                self.zero_grad(set_to_none=True)
            return

        for group in self.param_groups:
            scale = group.get("rho", self.rho) / (grad_norm + 1e-12)

            for p in group["params"]:
                if p.grad is None:
                    continue

                if group.get("adaptive", self.adaptive):
                    e_w = torch.pow(p, 2) * p.grad * scale
                else:
                    e_w = p.grad * scale

                p.add_(e_w)
                self.state[p]["e_w"] = e_w

        if zero_grad:
            self.zero_grad(set_to_none=True)

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                e_w = self.state[p].get("e_w", None)
                if e_w is not None:
                    p.sub_(e_w)

        if zero_grad:
            self.zero_grad(set_to_none=True)

    @torch.no_grad()
    def step(self, closure=None):
        raise NotImplementedError("Use first_step and second_step with SAM.")

    def zero_grad(self, set_to_none=True):
        self.base_optimizer.zero_grad(set_to_none=set_to_none)

In [ ]:
def kd_kl_loss(student_logits, teacher_logits, temperature=4.0):
    log_p = F.log_softmax(student_logits / temperature, dim=1)
    q = F.softmax(teacher_logits / temperature, dim=1)
    return F.kl_div(log_p, q, reduction="batchmean") * (temperature ** 2)

In [ ]:
def prepare_batch(images, labels, use_cutmix=False, cutmix_alpha=1.0):
    if use_cutmix:
        images, labels_a, labels_b, lam = apply_cutmix(images, labels, alpha=cutmix_alpha)
        return images, labels_a, labels_b, lam
    return images, labels, labels, 1.0


def forward_loss(
    model,
    images,
    labels,
    criterion,
    labels_a=None,
    labels_b=None,
    lam=1.0,
    use_amp=True,
    aux_weight=0.2,
    coarse_weight=0.0,
    fine_to_coarse_tensor=None,
    compact_weight=0.0,
    sep_weight=0.0,
    # ── MỚI: KD args ───────────────────────────────────────────
    teacher_model=None,
    kd_alpha=0.0,
    kd_temperature=4.0,
):
    amp_device = "cuda" if torch.cuda.is_available() else "cpu"
    uses_margin_head = getattr(model, "uses_margin_head", False)
    uses_aux_head = getattr(model, "uses_aux_head", False)
    uses_coarse_head = getattr(model, "uses_coarse_head", False)
 
    with torch.amp.autocast(amp_device, enabled=use_amp and torch.cuda.is_available()):
        if uses_margin_head:
            if labels_a is not None and labels_b is not None and lam != 1.0:
                outputs_a = model(images, target=labels_a)
                outputs_b = model(images, target=labels_b)
                loss = lam * criterion(outputs_a, labels_a) + (1.0 - lam) * criterion(outputs_b, labels_b)
                outputs = lam * outputs_a + (1.0 - lam) * outputs_b
            else:
                outputs = model(images, target=labels)
                loss = criterion(outputs, labels)
 
        elif uses_aux_head:
            main_outputs, aux_outputs = model(images, return_aux=True)
            if labels_a is not None and labels_b is not None and lam != 1.0:
                main_loss = lam * criterion(main_outputs, labels_a) + (1.0 - lam) * criterion(main_outputs, labels_b)
                aux_loss = lam * criterion(aux_outputs, labels_a) + (1.0 - lam) * criterion(aux_outputs, labels_b)
            else:
                main_loss = criterion(main_outputs, labels)
                aux_loss = criterion(aux_outputs, labels)
            loss = main_loss + aux_weight * aux_loss
            outputs = main_outputs
 
        elif uses_coarse_head:
            parts = model(images, return_parts=True)
            outputs = parts["logits"]
            coarse_outputs = parts["coarse_logits"]
            if labels_a is not None and labels_b is not None and lam != 1.0:
                fine_loss = lam * criterion(outputs, labels_a) + (1.0 - lam) * criterion(outputs, labels_b)
                if coarse_weight > 0.0 and fine_to_coarse_tensor is not None:
                    coarse_a = fine_to_coarse_tensor[labels_a]; coarse_b = fine_to_coarse_tensor[labels_b]
                    coarse_loss = lam * F.cross_entropy(coarse_outputs, coarse_a) + (1.0 - lam) * F.cross_entropy(coarse_outputs, coarse_b)
                else:
                    coarse_loss = torch.zeros((), device=outputs.device, dtype=outputs.dtype)
                if compact_weight > 0.0:
                    compact_a = embedding_compactness_loss(model, parts, labels_a)
                    compact_b = embedding_compactness_loss(model, parts, labels_b)
                    compact_loss = lam * compact_a + (1.0 - lam) * compact_b
                else:
                    compact_loss = torch.zeros((), device=outputs.device, dtype=outputs.dtype)
            else:
                fine_loss = criterion(outputs, labels)
                if coarse_weight > 0.0 and fine_to_coarse_tensor is not None:
                    coarse_labels = fine_to_coarse_tensor[labels]
                    coarse_loss = F.cross_entropy(coarse_outputs, coarse_labels)
                else:
                    coarse_loss = torch.zeros((), device=outputs.device, dtype=outputs.dtype)
                if compact_weight > 0.0:
                    compact_loss = embedding_compactness_loss(model, parts, labels)
                else:
                    compact_loss = torch.zeros((), device=outputs.device, dtype=outputs.dtype)
            if sep_weight > 0.0:
                sep_loss = prototype_separation_loss(model)
            else:
                sep_loss = torch.zeros((), device=outputs.device, dtype=outputs.dtype)
            loss = fine_loss + coarse_weight * coarse_loss + compact_weight * compact_loss + sep_weight * sep_loss
 
        elif getattr(model, 'uses_msda_head', False):
            if labels_a is not None and labels_b is not None and lam != 1.0:
                outputs, proto_loss = model(images, labels_a)
                main_loss = lam * criterion(outputs, labels_a) + (1.0 - lam) * criterion(outputs, labels_b)
            else:
                outputs, proto_loss = model(images, labels)
                main_loss = criterion(outputs, labels)
            loss = main_loss + aux_weight * proto_loss
 
        else:
            outputs = model(images)
            if labels_a is not None and labels_b is not None and lam != 1.0:
                loss = lam * criterion(outputs, labels_a) + (1.0 - lam) * criterion(outputs, labels_b)
            else:
                loss = criterion(outputs, labels)
 
        # ── MỚI: Logit Knowledge Distillation ──────────────────────────────
        # L = (1-alpha)*CE + alpha*KL(student||teacher)
        if teacher_model is not None and kd_alpha > 0.0:
            with torch.no_grad():
                t_out = teacher_model(images)              # teacher logits (frozen)
            kd = kd_kl_loss(outputs, t_out, temperature=kd_temperature)
            loss = (1.0 - kd_alpha) * loss + kd_alpha * kd
 
    return outputs, loss


def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device,
    use_cutmix=False,
    cutmix_alpha=1.0,
    use_sam=False,
    aux_weight=0.2,
    coarse_weight=0.0,
    fine_to_coarse_tensor=None,
    compact_weight=0.0,
    sep_weight=0.0,
    teacher_model=None,
    kd_alpha=0.0,
    kd_temperature=4.0,
):
    model.train()

    running_loss = 0.0
    correct = 0.0
    total = 0

    pbar = tqdm(loader, leave=False)

    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        images, labels_a, labels_b, lam = prepare_batch(
            images=images,
            labels=labels,
            use_cutmix=use_cutmix,
            cutmix_alpha=cutmix_alpha
        )

        optimizer.zero_grad(set_to_none=True)

        if not use_sam:
            outputs, loss = forward_loss(
                model=model,
                images=images,
                labels=labels,
                criterion=criterion,
                labels_a=labels_a if use_cutmix else None,
                labels_b=labels_b if use_cutmix else None,
                lam=lam,
                use_amp=True,
                aux_weight=aux_weight,
                coarse_weight=coarse_weight,
                fine_to_coarse_tensor=fine_to_coarse_tensor,
                compact_weight=compact_weight,
                sep_weight=sep_weight,
                teacher_model=teacher_model,
                kd_alpha=kd_alpha,
                kd_temperature=kd_temperature,
            )

            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss detected: {loss.item()}")

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)

            if use_cutmix:
                correct += lam * (preds == labels_a).sum().item() + (1.0 - lam) * (preds == labels_b).sum().item()
            else:
                correct += (preds == labels).sum().item()

            total += labels.size(0)
            pbar.set_postfix(loss=running_loss / total, acc=correct / total)
            continue

        outputs_1, loss_1 = forward_loss(
            model=model,
            images=images,
            labels=labels,
            criterion=criterion,
            labels_a=labels_a if use_cutmix else None,
            labels_b=labels_b if use_cutmix else None,
            lam=lam,
            use_amp=False,
            aux_weight=aux_weight,
            coarse_weight=coarse_weight,
            fine_to_coarse_tensor=fine_to_coarse_tensor,
            compact_weight=compact_weight,
            sep_weight=sep_weight,
            teacher_model=teacher_model,
            kd_alpha=kd_alpha,
            kd_temperature=kd_temperature,
        )

        if not torch.isfinite(loss_1):
            raise RuntimeError(f"Non-finite loss detected at SAM first pass: {loss_1.item()}")

        loss_1.backward()
        optimizer.first_step(zero_grad=True)

        _, loss_2 = forward_loss(
            model=model,
            images=images,
            labels=labels,
            criterion=criterion,
            labels_a=labels_a if use_cutmix else None,
            labels_b=labels_b if use_cutmix else None,
            lam=lam,
            use_amp=False,
            aux_weight=aux_weight,
            coarse_weight=coarse_weight,
            fine_to_coarse_tensor=fine_to_coarse_tensor,
            compact_weight=compact_weight,
            sep_weight=sep_weight,
            teacher_model=teacher_model,
            kd_alpha=kd_alpha,
            kd_temperature=kd_temperature,
        )

        if not torch.isfinite(loss_2):
            raise RuntimeError(f"Non-finite loss detected at SAM second pass: {loss_2.item()}")

        loss_2.backward()
        optimizer.second_step(zero_grad=False)
        optimizer.base_optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        running_loss += loss_1.item() * images.size(0)
        preds = outputs_1.argmax(dim=1)

        if use_cutmix:
            correct += lam * (preds == labels_a).sum().item() + (1.0 - lam) * (preds == labels_b).sum().item()
        else:
            correct += (preds == labels).sum().item()

        total += labels.size(0)
        pbar.set_postfix(loss=running_loss / total, acc=correct / total)

    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    amp_device = "cuda" if torch.cuda.is_available() else "cpu"

    with torch.no_grad():
        pbar = tqdm(loader, leave=False)

        for images, labels in pbar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast(amp_device, enabled=torch.cuda.is_available()):
                if getattr(model, 'uses_msda_head', False):
                    outputs, _ = model(images)  # discard proto_loss at eval
                else:
                    outputs = model(images)
                loss = criterion(outputs, labels)

            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite eval loss detected: {loss.item()}")

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

            pbar.set_postfix(loss=running_loss / total, acc=correct / total)

    return running_loss / total, correct / total, np.array(all_labels), np.array(all_preds)

In [ ]:
# Coarse experimental heads are outside the reported audit architectures.
fine_to_coarse_tensor = None
fine_to_coarse_list = None
num_coarse_classes = 0


# Model/optimizer/scheduler utilities

In [ ]:
# =====================================================================
# Model / optimizer / scheduler utilities
# =====================================================================
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    balanced_accuracy_score,
)


def build_model(
    model_name="efficientnet_b2",
    model_variant="original",
    drop_rate=0.0,
    pretrained=True,
    fusion_channels=512,
    embedding_dim=256,
    arcface_s=24.0,
    arcface_m=0.15,
    region_grid=2,
    use_gem=True,
    attn_hidden=128,
    refine_kernel_size=7,
    use_cosine_classifier=False,
    cosine_s=24.0,
    top_k=3,
    score_hidden=128,
    mamba_dim=96,
    d_state=16,
    d_conv=3,
    expand=2,
    init_alpha=0.1,
    use_external_mamba=False,
    prototype_scale=12.0,
    classifier_scale=1.0,
    uncertainty_hidden=128,
    num_coarse_classes=0,
    fine_to_coarse=None,
    residual_scale=0.5,
    coarse_scale=12.0,
    gate_temperature=1.0,
    # ── MSDA params ─────────────────────
    bifpn_dim=128,
    proto_temperature=0.07,
    msda_transformer_depth=2,
    msda_freeze_stages=2,
):
    if model_name == "efficientnet_b2" and model_variant not in {"original", "dual_scale"}:
        raise ValueError("This public template supports B2 original/dual_scale, B0 and ViT only.")
    if model_name != "efficientnet_b2":
        model = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=num_classes,
            drop_rate=drop_rate
        )
    else:
        if model_variant == "original":
            model = efficientnet_b2_original(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate
            )
        elif model_variant == "dual_scale":
            model = efficientnet_b2_dual_scale(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                fusion_channels=fusion_channels
            )
        elif model_variant == "dual_scale_arcface":
            model = efficientnet_b2_dual_scale_arcface(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                fusion_channels=fusion_channels,
                embedding_dim=embedding_dim,
                arcface_s=arcface_s,
                arcface_m=arcface_m
            )
        elif model_variant == "arcface":
            model = efficientnet_b2_arcface(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                embedding_dim=embedding_dim,
                arcface_s=arcface_s,
                arcface_m=arcface_m
            )
        elif model_variant == "regional":
            model = efficientnet_b2_regional(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                embedding_dim=embedding_dim,
                region_grid=region_grid,
                use_gem=use_gem,
                attn_hidden=attn_hidden
            )
        elif model_variant == "spatial_refine":
            model = efficientnet_b2_spatial_refine(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                refine_kernel_size=refine_kernel_size
            )
        elif model_variant == "compact_head":
            model = efficientnet_b2_compact_head(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                embedding_dim=embedding_dim,
                use_gem=use_gem,
                cosine_s=cosine_s,
                use_cosine_classifier=use_cosine_classifier
            )
        elif model_variant == "dual_evidence":
            model = efficientnet_b2_dual_evidence(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                embedding_dim=embedding_dim,
                region_grid=region_grid,
                top_k=top_k,
                use_gem=use_gem,
                score_hidden=score_hidden
            )
        elif model_variant == "c4_refine":
            model = efficientnet_b2_c4_refine(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate
            )
        elif model_variant == "mamba_refine":
            model = efficientnet_b2_mamba_refine(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                mamba_dim=mamba_dim,
                d_state=d_state,
                d_conv=d_conv,
                expand=expand,
                init_alpha=init_alpha,
                use_external_mamba=use_external_mamba
            )
        elif model_variant == "upg":
            model = efficientnet_b2_upg(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                embedding_dim=embedding_dim,
                prototype_scale=prototype_scale,
                classifier_scale=classifier_scale,
                uncertainty_hidden=uncertainty_hidden,
                use_gem=use_gem,
                num_coarse_classes=num_coarse_classes,
                fine_to_coarse=fine_to_coarse,
                residual_scale=residual_scale,
                coarse_scale=coarse_scale,
                gate_temperature=gate_temperature
            )
        elif model_variant == 'msda_tada':
            model = EfficientSkin_TADA(
                num_classes=num_classes, drop_rate=drop_rate, pretrained=pretrained)
        elif model_variant == 'msda_bifpn':
            model = EfficientSkin_BiFPN(
                num_classes=num_classes, D=bifpn_dim,
                drop_rate=drop_rate, pretrained=pretrained)
        elif model_variant == 'msda_lwt':
            model = EfficientSkin_LWT(
                num_classes=num_classes, transformer_dim=bifpn_dim,
                drop_rate=drop_rate, pretrained=pretrained)
        elif model_variant in ('msda_proto', 'msda_full'):
            if model_variant == 'msda_proto':
                model = EfficientSkin_Proto(
                    num_classes=num_classes, drop_rate=drop_rate,
                    proto_temperature=proto_temperature, pretrained=pretrained)
            else:
                model = EfficientSkinMSDA(
                    num_classes=num_classes, D=bifpn_dim,
                    transformer_depth=msda_transformer_depth,
                    drop_rate=drop_rate, proto_temperature=proto_temperature,
                    pretrained=pretrained, freeze_backbone_stages=msda_freeze_stages)
            model.uses_msda_head = True  # flag for forward_loss / evaluate
        elif model_variant == "msda_tada_proto":
            model = EfficientSkin_TADA_Proto(
                num_classes=num_classes, drop_rate=drop_rate,
                proto_temperature=proto_temperature, pretrained=pretrained)
            model.uses_msda_head = True

        elif model_variant == "msda_tada_gem":
            model = EfficientSkin_TADA_GeM(
                num_classes=num_classes, drop_rate=drop_rate, pretrained=pretrained)

        elif model_variant == "msda_tada_proto_gem":
            model = EfficientSkin_TADA_Proto_GeM(
                num_classes=num_classes, drop_rate=drop_rate,
                proto_temperature=proto_temperature, pretrained=pretrained)
            model.uses_msda_head = True

        elif model_variant == "msda_full_fixed":
            model = EfficientSkinMSDA_Fixed(
                num_classes=num_classes, D=bifpn_dim,
                transformer_depth=msda_transformer_depth,
                drop_rate=drop_rate, proto_temperature=proto_temperature,
                pretrained=pretrained, freeze_backbone_stages=msda_freeze_stages)
            model.uses_msda_head = True
        elif model_variant == "wagf":
            model = WAGFFusion_EfficientNetB2(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                learnable_fusion=False,)
        elif model_variant == "eca":
            model = AttnEfficientNet(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                attn="eca",
            )

        elif model_variant == "cbam":
            model = AttnEfficientNet(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                attn="cbam",
            )
        elif model_variant == "mssf":
            model = MSSF_EfficientNet(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                fuse_dim=256,
                out_indices=(2, 3, 4),   # 28×28, 14×14, 7×7
            )
        else:
            raise ValueError(f"Unsupported model_variant: {model_variant}")

    return model.to(device)


def build_optimizer(model, optimizer_name, lr_value=None, use_sam=False, sam_rho=0.05, sam_adaptive=False):
    actual_lr = lr if lr_value is None else lr_value

    if not use_sam:
        if optimizer_name == "adam":
            return torch.optim.Adam(model.parameters(), lr=actual_lr, weight_decay=weight_decay)
        if optimizer_name == "adamw":
            return torch.optim.AdamW(model.parameters(), lr=actual_lr, weight_decay=weight_decay)
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    if optimizer_name == "adam":
        return SAM(
            model.parameters(),
            torch.optim.Adam,
            rho=sam_rho,
            adaptive=sam_adaptive,
            lr=actual_lr,
            weight_decay=weight_decay
        )

    if optimizer_name == "adamw":
        return SAM(
            model.parameters(),
            torch.optim.AdamW,
            rho=sam_rho,
            adaptive=sam_adaptive,
            lr=actual_lr,
            weight_decay=weight_decay
        )

    raise ValueError(f"Unsupported optimizer: {optimizer_name}")


def build_scheduler(optimizer, scheduler_name):
    target_optimizer = optimizer.base_optimizer if hasattr(optimizer, "base_optimizer") else optimizer

    if scheduler_name == "none":
        return None
    if scheduler_name == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(target_optimizer, T_max=epochs)
    raise ValueError(f"Unsupported scheduler: {scheduler_name}")


def slugify_exp_name(exp_name):
    return exp_name.replace(" ", "_").replace("+", "plus").lower()


def save_history_files(exp_name, history):
    exp_slug = slugify_exp_name(exp_name)
    hist_df = pd.DataFrame(history)
    csv_path = history_dir / f"{exp_slug}_history.csv"
    json_path = history_dir / f"{exp_slug}_history.json"
    hist_df.to_csv(csv_path, index=False)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)
    return csv_path, json_path


# =====================================================================
# Metrics utilities
# =====================================================================
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
)


def compute_full_metrics(y_true, y_pred, num_classes, prefix=""):
    """Trả về dict metrics đầy đủ: acc, balanced_acc, macro P/R/F1, weighted F1."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    labels_idx = list(range(num_classes))

    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels_idx, average="macro", zero_division=0
    )

    return {
        f"{prefix}acc":             float((y_true == y_pred).mean()),
        f"{prefix}balanced_acc":    float(balanced_accuracy_score(y_true, y_pred)),
        f"{prefix}macro_f1":        float(f1),
        f"{prefix}macro_precision": float(prec),
        f"{prefix}macro_recall":    float(rec),
        f"{prefix}weighted_f1":     float(
            f1_score(y_true, y_pred, labels=labels_idx,
                     average="weighted", zero_division=0)
        ),
    }


# Cột hiển thị ưu tiên trong bảng summary
SUMMARY_LEAD_COLS = [
    "experiment", "model_name", "model_variant", "status",
    "best_epoch",
    "best_val_acc", "best_val_balanced_acc", "best_val_macro_f1",
    "test_acc", "test_balanced_acc", "test_macro_f1",
    "test_weighted_f1", "test_macro_precision", "test_macro_recall",
    "test_loss",
]


def order_summary_columns(df):
    lead = [c for c in SUMMARY_LEAD_COLS if c in df.columns]
    rest = [c for c in df.columns if c not in lead]
    return df[lead + rest]


def save_summary_files(all_results):
    summary_df = order_summary_columns(pd.DataFrame(all_results))
    csv_path = work_dir / "experiment_summary.csv"
    json_path = work_dir / "experiment_summary.json"
    summary_df.to_csv(csv_path, index=False)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2, default=str)
    return summary_df, csv_path, json_path


def run_experiment(
    exp_name,
    optimizer_name,
    scheduler_name,
    model_name="efficientnet_b2",
    model_variant="original",
    lr_value=None,
    label_smoothing=0.0,
    use_cutmix=False,
    cutmix_alpha=1.0,
    use_sam=False,
    sam_rho=0.05,
    sam_adaptive=False,
    drop_rate=0.0,
    fusion_channels=512,
    embedding_dim=256,
    arcface_s=30.0,
    arcface_m=0.3,
    aux_weight=0.2,
    region_grid=2,
    use_gem=True,
    attn_hidden=128,
    refine_kernel_size=7,
    use_cosine_classifier=False,
    cosine_s=24.0,
    top_k=3,
    score_hidden=128,
    mamba_dim=96,
    d_state=16,
    d_conv=3,
    expand=2,
    init_alpha=0.1,
    use_external_mamba=False,
    prototype_scale=12.0,
    classifier_scale=1.0,
    uncertainty_hidden=128,
    use_kd=False,
    kd_teacher_path=None,
    kd_alpha=0.3,
    kd_temperature=4.0,
    kd_teacher_variant="dual_scale",
    kd_teacher_fusion_channels=512,
    num_coarse_classes=0,
    fine_to_coarse=None,
    residual_scale=0.5,
    coarse_scale=12.0,
    coarse_weight=0.0,
    compact_weight=0.0,
    sep_weight=0.0,
    gate_temperature=1.0,
    run_diagnostic=False,
    baseline_checkpoint=None,
    bifpn_dim=128,
    proto_temperature=0.07,
    msda_transformer_depth=2,
    msda_freeze_stages=2,
    # ── MỚI: chọn checkpoint theo metric nào trên VAL ──────────────
    select_metric="val_acc",   # "val_acc" | "val_macro_f1" | "val_balanced_acc"
):
    if use_kd and kd_teacher_path is None:
        raise ValueError("use_kd=True requires an explicit teacher checkpoint")
    if use_kd and not Path(kd_teacher_path).is_file():
        raise FileNotFoundError(f"Teacher checkpoint not found: {kd_teacher_path}")
    reset_experiment_seed(seed)
    run_train_loader, run_val_loader, run_test_loader = \
        build_fresh_run_loaders()
    is_dev_mode = (DATA_MODE == "dev")
    is_final_mode = (DATA_MODE == "final")
    actual_lr = lr if lr_value is None else lr_value
    exp_slug = slugify_exp_name(exp_name)
    if is_dev_mode:
        save_path = work_dir / f"{exp_slug}_best.pth"
    else:
        save_path = work_dir / f"{exp_slug}_final.pth"

    print("=" * 90)
    print(f"Running: {exp_name}")
    print(f"  model={model_name} | variant={model_variant} | lr={actual_lr} | "
          f"sam={use_sam}({sam_rho}) | cutmix={use_cutmix} | ls={label_smoothing} | "
          f"select_by={select_metric}")
    n_val = len(val_dataset) if val_dataset is not None else 0

    print(
        f"  data_mode={DATA_MODE} | "
        f"train={len(train_dataset)} | "
        f"val={n_val} | "
        f"test={len(test_dataset)}"
    )
    print("=" * 90)

    model_kwargs = dict(
        model_name=model_name,
        model_variant=model_variant,
        drop_rate=drop_rate,
        fusion_channels=fusion_channels,
        embedding_dim=embedding_dim,
        arcface_s=arcface_s,
        arcface_m=arcface_m,
        region_grid=region_grid,
        use_gem=use_gem,
        attn_hidden=attn_hidden,
        refine_kernel_size=refine_kernel_size,
        use_cosine_classifier=use_cosine_classifier,
        cosine_s=cosine_s,
        top_k=top_k,
        score_hidden=score_hidden,
        mamba_dim=mamba_dim,
        d_state=d_state,
        d_conv=d_conv,
        expand=expand,
        init_alpha=init_alpha,
        use_external_mamba=use_external_mamba,
        prototype_scale=prototype_scale,
        classifier_scale=classifier_scale,
        uncertainty_hidden=uncertainty_hidden,
        num_coarse_classes=num_coarse_classes,
        fine_to_coarse=fine_to_coarse,
        residual_scale=residual_scale,
        coarse_scale=coarse_scale,
        gate_temperature=gate_temperature,
        bifpn_dim=bifpn_dim,
        proto_temperature=proto_temperature,
        msda_transformer_depth=msda_transformer_depth,
        msda_freeze_stages=msda_freeze_stages,
    )

    # ── config dùng chung cho history rows + summary row ──────────────
    config = {
        "experiment": exp_name,
        "model_name": model_name,
        "model_variant": model_variant,
        "optimizer": optimizer_name,
        "scheduler": scheduler_name,
        "lr": float(actual_lr),
        "label_smoothing": float(label_smoothing),
        "use_cutmix": bool(use_cutmix),
        "cutmix_alpha": float(cutmix_alpha),
        "use_sam": bool(use_sam),
        "sam_rho": float(sam_rho),
        "sam_adaptive": bool(sam_adaptive),
        "drop_rate": float(drop_rate),
        "fusion_channels": int(fusion_channels),
        "embedding_dim": int(embedding_dim),
        "arcface_s": float(arcface_s),
        "arcface_m": float(arcface_m),
        "aux_weight": float(aux_weight),
        "region_grid": int(region_grid),
        "top_k": int(top_k),
        "use_gem": bool(use_gem),
        "attn_hidden": int(attn_hidden),
        "score_hidden": int(score_hidden),
        "refine_kernel_size": int(refine_kernel_size),
        "use_cosine_classifier": bool(use_cosine_classifier),
        "cosine_s": float(cosine_s),
        "mamba_dim": int(mamba_dim),
        "d_state": int(d_state),
        "d_conv": int(d_conv),
        "expand": int(expand),
        "init_alpha": float(init_alpha),
        "use_external_mamba": bool(use_external_mamba),
        "prototype_scale": float(prototype_scale),
        "classifier_scale": float(classifier_scale),
        "uncertainty_hidden": int(uncertainty_hidden),
        "use_kd": bool(use_kd),
        "kd_teacher_path": str(kd_teacher_path) if kd_teacher_path is not None else None,
        "kd_alpha": float(kd_alpha),
        "kd_temperature": float(kd_temperature),
        "num_coarse_classes": int(num_coarse_classes or 0),
        "residual_scale": float(residual_scale),
        "coarse_scale": float(coarse_scale),
        "coarse_weight": float(coarse_weight),
        "compact_weight": float(compact_weight),
        "sep_weight": float(sep_weight),
        "gate_temperature": float(gate_temperature),
        "select_metric": select_metric,
        "epochs": int(epochs),
        "batch_size": int(batch_size),
        "img_size": int(img_size),
        "n_train": int(len(train_dataset)),
        "data_mode": DATA_MODE,
        "n_val": int(len(val_dataset)) if val_dataset is not None else 0,
        "n_test": int(len(test_dataset)),
    }

    model = build_model(pretrained=True, **model_kwargs)

    # ── Teacher (KD) ──────────────────────────────────────────────────
    teacher_model = None
    if use_kd and kd_teacher_path is not None:
        state = torch.load(kd_teacher_path, map_location=device, weights_only=True)
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        elif isinstance(state, dict) and "model" in state:
            state = state["model"]

        if kd_teacher_variant == "dual_scale":
            teacher_model = efficientnet_b2_dual_scale(
                num_classes=num_classes, pretrained=False,
                fusion_channels=kd_teacher_fusion_channels,
            )
        else:
            teacher_model = build_model(
                pretrained=False, model_name="efficientnet_b2",
                model_variant=kd_teacher_variant,
                fusion_channels=kd_teacher_fusion_channels,
            )

        kd_missing, kd_unexpected = teacher_model.load_state_dict(state, strict=False)
        if kd_missing or kd_unexpected:
            raise RuntimeError(f"Teacher architecture/checkpoint mismatch: missing={kd_missing}, unexpected={kd_unexpected}")
        print(f"[KD] Teacher loaded. missing={len(kd_missing)} unexpected={len(kd_unexpected)}")
        teacher_model = teacher_model.to(device).eval()
        for p in teacher_model.parameters():
            p.requires_grad = False

    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = build_optimizer(
        model=model, optimizer_name=optimizer_name, lr_value=lr_value,
        use_sam=use_sam, sam_rho=sam_rho, sam_adaptive=sam_adaptive,
    )
    scheduler = build_scheduler(optimizer, scheduler_name)

    amp_device = "cuda" if torch.cuda.is_available() else "cpu"
    scaler = torch.amp.GradScaler(amp_device, enabled=torch.cuda.is_available())

    best_score = -float("inf")
    best_epoch = 0
    best_val_metrics = {}
    history = []

    # =================================================================
    # TRAIN  +  VAL  (model selection chỉ dùng VAL)
    # =================================================================
    # ================================================================
    # TRAIN LOOP
    # ================================================================
    
    best_score = -float("inf")
    best_epoch = 0
    best_val_metrics = {}
    
    history = []
    
    # Final test criterion:
    # pure CE, chỉ để monitor/evaluate, KHÔNG backprop
    final_eval_criterion = (
        nn.CrossEntropyLoss()
        if is_final_mode
        else None
    )
    
    final_epoch_test_metrics = None
    final_epoch_test_loss = None
    final_y_true = None
    final_y_pred = None
    
    
    for epoch in range(1, epochs + 1):
    
        current_lr = optimizer.param_groups[0]["lr"]
    
        # ============================================================
        # 1. TRAIN — đây là chỗ DUY NHẤT update parameters
        # ============================================================
    
        train_loss, train_acc = train_one_epoch(
            model=model,
            loader=run_train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=device,
            use_cutmix=use_cutmix,
            cutmix_alpha=cutmix_alpha,
            use_sam=use_sam,
            aux_weight=aux_weight,
            coarse_weight=coarse_weight,
            fine_to_coarse_tensor=(
                fine_to_coarse_tensor
                if (
                    coarse_weight > 0.0
                    or compact_weight > 0.0
                )
                else None
            ),
            compact_weight=compact_weight,
            sep_weight=sep_weight,
            teacher_model=teacher_model,
            kd_alpha=kd_alpha,
            kd_temperature=kd_temperature,
        )
    
    
        # ============================================================
        # 2A. DEVELOPMENT: evaluate VAL + model selection
        # ============================================================
    
        if is_dev_mode:
    
            val_loss, val_acc, y_true_val, y_pred_val = evaluate(
                model=model,
                loader=run_val_loader,
                criterion=criterion,
                device=device
            )
    
            val_metrics = compute_full_metrics(
                y_true_val,
                y_pred_val,
                num_classes,
                prefix="val_"
            )
    
            val_metrics["val_loss"] = float(val_loss)
    
            row = {
                **config,
                "epoch": epoch,
                "lr_epoch": float(current_lr),
                "train_loss": float(train_loss),
                "train_acc": float(train_acc),
                **val_metrics,
            }
    
            score = val_metrics.get(
                select_metric,
                val_metrics["val_acc"]
            )
    
            # VALIDATION được phép chọn checkpoint
            if score > best_score:
    
                best_score = float(score)
                best_epoch = epoch
                best_val_metrics = dict(val_metrics)
    
                torch.save(
                    model.state_dict(),
                    save_path
                )
    
                print(
                    f"    ↳ new best "
                    f"({select_metric}={best_score:.4f})"
                )
    
            print(
                f"[{exp_name}] "
                f"Epoch {epoch:02d}/{epochs} | "
                f"lr={current_lr:.2e} | "
                f"train_loss={train_loss:.4f} "
                f"train_acc={train_acc:.4f} || "
                f"val_loss={val_metrics['val_loss']:.4f} "
                f"val_acc={val_metrics['val_acc']:.4f} "
                f"val_bal={val_metrics['val_balanced_acc']:.4f} "
                f"val_f1={val_metrics['val_macro_f1']:.4f}"
            )
    
    
        # ============================================================
        # 2B. FINAL: evaluate TEST — NEVER model selection
        # ============================================================
    
        else:
    
            epoch_test_metrics = {}
    
            if FINAL_TEST_EACH_EPOCH:
    
                test_loss_epoch, _, y_true_test, y_pred_test = evaluate(
                    model=model,
                    loader=run_test_loader,
                    criterion=final_eval_criterion,
                    device=device
                )
    
                epoch_test_metrics = compute_full_metrics(
                    y_true_test,
                    y_pred_test,
                    num_classes,
                    prefix="test_"
                )
    
                epoch_test_metrics["test_loss"] = float(
                    test_loss_epoch
                )
    
                # giữ lại epoch cuối
                if epoch == epochs:
                    final_epoch_test_metrics = dict(
                        epoch_test_metrics
                    )
    
                    final_epoch_test_loss = float(
                        test_loss_epoch
                    )
    
                    final_y_true = y_true_test
                    final_y_pred = y_pred_test
    
            row = {
                **config,
                "epoch": epoch,
                "lr_epoch": float(current_lr),
                "train_loss": float(train_loss),
                "train_acc": float(train_acc),
                **epoch_test_metrics,
            }
    
            if FINAL_TEST_EACH_EPOCH:
    
                print(
                    f"[{exp_name}] "
                    f"Epoch {epoch:02d}/{epochs} | "
                    f"lr={current_lr:.2e} | "
                    f"train_loss={train_loss:.4f} "
                    f"train_acc={train_acc:.4f} || "
                    f"test_loss={epoch_test_metrics['test_loss']:.4f} "
                    f"test_acc={epoch_test_metrics['test_acc']:.4f} "
                    f"test_bal={epoch_test_metrics['test_balanced_acc']:.4f} "
                    f"test_f1={epoch_test_metrics['test_macro_f1']:.4f}"
                )
    
            else:
    
                print(
                    f"[{exp_name}] "
                    f"Epoch {epoch:02d}/{epochs} | "
                    f"lr={current_lr:.2e} | "
                    f"train_loss={train_loss:.4f} "
                    f"train_acc={train_acc:.4f}"
                )
    
    
        # ============================================================
        # Scheduler hoàn toàn độc lập metric
        # ============================================================
    
        if scheduler is not None:
            scheduler.step()
    
    
        history.append(row)
    
        save_history_files(
            exp_name,
            history
        )
    
    
    # ================================================================
    # END TRAINING
    # ================================================================
    
    if is_dev_mode:
    
        print(
            f"[{exp_name}] "
            f"Best VAL {select_metric} = "
            f"{best_score:.4f} @ epoch {best_epoch}"
        )
    
    else:
    
        # Final model = EXACTLY last epoch
        # KHÔNG dựa vào test score
        best_epoch = epochs
    
        torch.save(
            model.state_dict(),
            save_path
        )
    
        print(
            f"[{exp_name}] FINAL MODEL SAVED "
            f"@ fixed epoch {epochs}"
        )

    # ── giải phóng bộ nhớ trước khi load checkpoint để test ───────────
    del criterion, optimizer, scheduler, scaler, model
    if teacher_model is not None:
        del teacher_model
        teacher_model = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if final_eval_criterion is not None:
        del final_eval_criterion
    # =================================================================
    # TEST — chạy đúng 1 lần trên checkpoint best-val
    # =================================================================
    # ================================================================
    # DEV / FINAL EVALUATION
    # ================================================================
    
    test_loss = None
    
    test_metrics = {
        f"test_{k}": None
        for k in [
            "acc",
            "balanced_acc",
            "macro_f1",
            "macro_precision",
            "macro_recall",
            "weighted_f1",
        ]
    }
    
    test_report_path = None
    test_confusion_path = None
    
    y_true_test = None
    y_pred_test = None
    
    
    # ================================================================
    # DEV:
    # evaluate BEST-VALIDATION checkpoint
    # ================================================================
    
    if is_dev_mode:
    
        if best_epoch > 0 and save_path.exists():
    
            best_model = build_model(
                pretrained=False,
                **model_kwargs
            )
    
            best_model.load_state_dict(
                torch.load(
                    save_path,
                    map_location=device
                ),
                strict=True
            )
    
            test_criterion = nn.CrossEntropyLoss()
    
            test_loss, _, y_true_test, y_pred_test = evaluate(
                model=best_model,
                loader=run_test_loader,
                criterion=test_criterion,
                device=device
            )
    
            test_loss = float(test_loss)
    
            test_metrics = compute_full_metrics(
                y_true_test,
                y_pred_test,
                num_classes,
                prefix="test_"
            )
    
            del best_model, test_criterion
    
        else:
            print(
                f"[{exp_name}] WARNING: "
                f"không có best-validation checkpoint -> bỏ qua test."
            )
    
    
    # ================================================================
    # FINAL:
    # evaluate FIXED LAST-EPOCH checkpoint
    # ================================================================
    
    else:
    
        if not save_path.exists():
            raise RuntimeError(
                f"Final checkpoint không tồn tại: {save_path}"
            )
    
        final_model = build_model(
            pretrained=False,
            **model_kwargs
        )
    
        final_model.load_state_dict(
            torch.load(
                save_path,
                map_location=device
            ),
            strict=True
        )
    
        test_criterion = nn.CrossEntropyLoss()
    
        test_loss, _, y_true_test, y_pred_test = evaluate(
            model=final_model,
            loader=run_test_loader,
            criterion=test_criterion,
            device=device
        )
    
        test_loss = float(test_loss)
    
        test_metrics = compute_full_metrics(
            y_true_test,
            y_pred_test,
            num_classes,
            prefix="test_"
        )
    
        print("-" * 90)
        print(
            f"[{exp_name}] FINAL TEST "
            f"(fixed epoch {epochs})"
        )
        print(f"  acc           = {test_metrics['test_acc']:.4f}")
        print(f"  balanced_acc  = {test_metrics['test_balanced_acc']:.4f}")
        print(f"  macro_f1      = {test_metrics['test_macro_f1']:.4f}")
        print(f"  macro_prec    = {test_metrics['test_macro_precision']:.4f}")
        print(f"  macro_recall  = {test_metrics['test_macro_recall']:.4f}")
        print(f"  weighted_f1   = {test_metrics['test_weighted_f1']:.4f}")
        print(f"  loss          = {test_loss:.4f}")
        print("-" * 90)
    
        del final_model, test_criterion
    
    
    # ================================================================
    # SAVE TEST REPORT / CONFUSION
    # dùng chung cho DEV và FINAL
    # ================================================================
    
    if y_true_test is not None and y_pred_test is not None:
    
        all_labels_idx = list(range(num_classes))
    
        print(
            classification_report(
                y_true_test,
                y_pred_test,
                labels=all_labels_idx,
                target_names=class_names,
                digits=4,
                zero_division=0
            )
        )
    
        report = classification_report(
            y_true_test,
            y_pred_test,
            labels=all_labels_idx,
            target_names=class_names,
            digits=4,
            zero_division=0,
            output_dict=True
        )
    
        test_report_path = (
            work_dir /
            f"{exp_slug}_test_report.json"
        )
    
        with open(
            test_report_path,
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                report,
                f,
                ensure_ascii=False,
                indent=2
            )
    
        test_confusion_path = (
            work_dir /
            f"{exp_slug}_test_confusion.npy"
        )
    
        np.save(
            test_confusion_path,
            confusion_matrix(
                y_true_test,
                y_pred_test,
                labels=all_labels_idx
            )
        )
    
    
    gc.collect()
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ── diagnostic (chỉ chạy nếu hàm tồn tại) ────────────────────────
    diagnostic_result = None
    if run_diagnostic and save_path.exists():
        if "evaluate_checkpoint_after_training" in globals():
            model_kwargs_for_diag = {}
            if model_variant == "upg":
                model_kwargs_for_diag = {
                    "embedding_dim": embedding_dim,
                    "prototype_scale": prototype_scale,
                    "classifier_scale": classifier_scale,
                    "uncertainty_hidden": uncertainty_hidden,
                    "use_gem": use_gem,
                    "num_coarse_classes": num_coarse_classes,
                    "fine_to_coarse": fine_to_coarse,
                    "residual_scale": residual_scale,
                    "coarse_scale": coarse_scale,
                    "gate_temperature": gate_temperature,
                }
            diagnostic_result = evaluate_checkpoint_after_training(
                checkpoint_path=save_path, exp_name=exp_name,
                model_name=model_name, model_variant=model_variant,
                model_kwargs=model_kwargs_for_diag,
                baseline_checkpoint=baseline_checkpoint,
            )
        else:
            print("[WARN] evaluate_checkpoint_after_training chưa được định nghĩa -> bỏ qua diagnostic.")

    # ── kết quả trả về ────────────────────────────────────────────────
    result = {
        **config,
    
        # ============================================================
        # DATA PROTOCOL
        # ============================================================
        "data_mode": DATA_MODE,
    
        # DEV:
        # epoch được chọn bằng validation
        #
        # FINAL:
        # không tồn tại "best epoch", vì model cuối là fixed epoch
        "best_epoch": (
            int(best_epoch)
            if is_dev_mode and best_epoch > 0
            else None
        ),
    
        # Chỉ FINAL mới có final_epoch
        "final_epoch": (
            int(epochs)
            if is_final_mode
            else None
        ),
    
        # ============================================================
        # VALIDATION METRICS
        # CHỈ có ý nghĩa trong DEV mode
        # ============================================================
        "best_val_score": (
            float(best_score)
            if is_dev_mode and best_epoch > 0
            else None
        ),
    
        "best_val_acc": (
            best_val_metrics.get("val_acc")
            if is_dev_mode
            else None
        ),
    
        "best_val_balanced_acc": (
            best_val_metrics.get("val_balanced_acc")
            if is_dev_mode
            else None
        ),
    
        "best_val_macro_f1": (
            best_val_metrics.get("val_macro_f1")
            if is_dev_mode
            else None
        ),
    
        "best_val_weighted_f1": (
            best_val_metrics.get("val_weighted_f1")
            if is_dev_mode
            else None
        ),
    
        "best_val_loss": (
            best_val_metrics.get("val_loss")
            if is_dev_mode
            else None
        ),
    
        # ============================================================
        # TEST METRICS
        #
        # DEV:
        #   test của best-validation checkpoint
        #
        # FINAL:
        #   test của fixed last-epoch checkpoint
        # ============================================================
        **test_metrics,
    
        "test_loss": (
            float(test_loss)
            if test_loss is not None
            else None
        ),
    
        # ============================================================
        # MODEL SELECTION INFO
        # ============================================================
        "model_selection": (
            f"best_{select_metric}"
            if is_dev_mode
            else "fixed_last_epoch"
        ),
    
        # ============================================================
        # LEGACY COMPATIBILITY
        #
        # best_acc trước đây = best validation accuracy.
        # Không được gán test_acc vào đây trong FINAL.
        # ============================================================
        "best_acc": (
            best_val_metrics.get("val_acc")
            if is_dev_mode
            else None
        ),
    
        # ============================================================
        # DIAGNOSTIC
        # ============================================================
        "run_diagnostic": bool(run_diagnostic),
    
        "baseline_checkpoint": (
            str(baseline_checkpoint)
            if baseline_checkpoint
            else None
        ),
    
        "diagnostic_json": (
            str(work_dir / f"{exp_slug}_diagnostic.json")
            if diagnostic_result is not None
            else None
        ),
    
        # ============================================================
        # OUTPUT FILES
        # ============================================================
        "test_report_path": (
            str(test_report_path)
            if test_report_path
            else None
        ),
    
        "test_confusion_path": (
            str(test_confusion_path)
            if test_confusion_path
            else None
        ),
    
        "save_path": str(save_path),
    
        "status": "ok",
    }
    del run_train_loader
    del run_val_loader
    del run_test_loader

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

Historical PAD external-evaluation cell omitted from this portable training template.
It runs inference against user-specific checkpoints and assumes 31 outputs.
For the reported external analysis, use `scripts/pad_eval.py` and the documented image/weight inputs.
The original cell remains in the unmodified archived notebook.


# Analysis utilities

# Load best checkpoint for diagnosis

# Analyze train set

# Refined training dataset

In [ ]:
# ================================================================
# DATASET PROTOCOL
#
# DATA_MODE = "dev"
#   original 80% train -> 70% train + 10% val
#   original 20% test  -> giữ nguyên
#
# DATA_MODE = "final"
#   dùng lại toàn bộ original 80% để train
#   original 20% test giữ nguyên
#   KHÔNG có validation
# ================================================================

exclude_paths = []


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


g = torch.Generator()
g.manual_seed(seed)


def build_skin31_train_sets(excluded_paths):
    """
    Tạo 7 phiên bản train giống protocol cũ.
    excluded_paths:
        dev   -> val_paths
        final -> empty set
    """

    return [
        FilteredImageFolder(
            skin31_train_dir,
            transform=orig_tf,
            exclude_paths=excluded_paths
        ),

        FilteredImageFolder(
            skin31_train_dir,
            transform=zoom_tf,
            exclude_paths=excluded_paths
        ),

        FilteredImageFolder(
            skin31_train_dir,
            transform=rot_tf,
            exclude_paths=excluded_paths
        ),

        FilteredImageFolder(
            skin31_train_dir,
            transform=bright_tf,
            exclude_paths=excluded_paths
        ),

        FilteredImageFolder(
            skin31_train_dir,
            transform=shear_tf,
            exclude_paths=excluded_paths
        ),

        FilteredImageFolder(
            skin31_train_dir,
            transform=vflip_tf,
            exclude_paths=excluded_paths
        ),

        FilteredImageFolder(
            skin31_train_dir,
            transform=hflip_tf,
            exclude_paths=excluded_paths
        ),
    ]


if dataset_name == "skin31":

    full_train_base = datasets.ImageFolder(skin31_train_dir)

    all_paths = np.array([
        path for path, _ in full_train_base.samples
    ])

    all_targets = np.array(full_train_base.targets)

    # ============================================================
    # DEVELOPMENT MODE: 70 / 10 / 20
    # ============================================================

    if DATA_MODE == "dev":

        train_paths, val_paths = train_test_split(
            all_paths,
            test_size=0.125,       # 12.5% của original 80% = 10% total
            random_state=seed,
            stratify=all_targets
        )

        train_paths = set(map(str, train_paths))
        val_paths = set(map(str, val_paths))

        # Train không chứa validation
        train_excluded_paths = val_paths

        train_sets = build_skin31_train_sets(
            excluded_paths=train_excluded_paths
        )

        train_dataset = ConcatDataset(train_sets)

        # Validation không augmentation
        val_dataset = FilteredImageFolder(
            skin31_train_dir,
            transform=to_tensor_norm,
            exclude_paths=train_paths
        )

        # Original raw train subset để tính class weights
        base_train_refined = FilteredImageFolder(
            skin31_train_dir,
            transform=None,
            exclude_paths=val_paths
        )

    # ============================================================
    # FINAL MODE: 80 / 20
    # ============================================================

    elif DATA_MODE == "final":

        # Không loại bất kỳ sample nào khỏi original 80% train
        train_excluded_paths = set()

        train_sets = build_skin31_train_sets(
            excluded_paths=train_excluded_paths
        )

        train_dataset = ConcatDataset(train_sets)

        # Final training KHÔNG còn validation
        val_dataset = None

        # Toàn bộ original 80% train dùng để tính weights
        base_train_refined = FilteredImageFolder(
            skin31_train_dir,
            transform=None,
            exclude_paths=[]
        )

    else:
        raise ValueError(
            f"DATA_MODE phải là 'dev' hoặc 'final', nhận được: {DATA_MODE}"
        )

    # ============================================================
    # TEST luôn giữ nguyên original 20%
    # ============================================================

    test_dataset = datasets.ImageFolder(
        skin31_test_dir,
        transform=to_tensor_norm
    )

    base_targets = np.array(base_train_refined.targets)


elif dataset_name == "ham10000":

    raise NotImplementedError(
        "DATA_MODE dev/final hiện được cấu hình cho Skin31."
    )

else:
    raise ValueError(f"Unsupported dataset_name: {dataset_name}")


# ================================================================
# CLASS BALANCING
# Recompute theo train subset thực sự đang dùng
# ================================================================

class_counts = np.bincount(
    base_targets,
    minlength=num_classes
)

class_weights = np.zeros_like(
    class_counts,
    dtype=np.float64
)

nonzero_mask = class_counts > 0

class_weights[nonzero_mask] = (
    1.0 / class_counts[nonzero_mask]
)

sample_weights = class_weights[base_targets]

# train_dataset gồm 7 deterministic variants
sample_weights = np.tile(
    sample_weights,
    len(train_sets)
)

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)


# ================================================================
# DATALOADER
# ================================================================

loader_kwargs = dict(
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=(num_workers > 0),
    prefetch_factor=(2 if num_workers > 0 else None),
    worker_init_fn=(seed_worker if num_workers > 0 else None),
)


# ================================================================
# FRESH LOADERS PER EXPERIMENT
# Giữ fix reproducibility đã xác nhận trước đó
# ================================================================

def build_fresh_run_loaders():

    # ============================================================
    # TRAIN LOADER
    #
    # Giữ NGUYÊN protocol training cũ:
    # - WeightedRandomSampler không có generator riêng
    # - DataLoader dùng g
    # ============================================================

    run_train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=sampler,
        drop_last=True,
        generator=g,
        **loader_kwargs
    )


    # ============================================================
    # VALIDATION LOADER
    # chỉ tồn tại ở DEV mode
    # ============================================================

    if val_dataset is not None:

        run_val_loader = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            **loader_kwargs
        )

    else:
        run_val_loader = None


    # ============================================================
    # TEST RNG RIÊNG CHO TỪNG EXPERIMENT
    #
    # Mục đích:
    # test mỗi epoch KHÔNG consume RNG của training.
    #
    # Generator này được tạo FRESH mỗi run
    # => run #1 / #2 / #3 không carry state cho nhau.
    # ============================================================

    run_test_g = torch.Generator()
    run_test_g.manual_seed(seed + 1000)


    # ============================================================
    # TEST LOADER
    # ============================================================

    run_test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,

        # Chỉ tách RNG ở FINAL.
        # DEV giữ nguyên protocol cũ.
        generator=(
            run_test_g
            if DATA_MODE == "final"
            else None
        ),

        **loader_kwargs
    )


    return (
        run_train_loader,
        run_val_loader,
        run_test_loader
    )


# ================================================================
# SANITY CHECK
# ================================================================

print("=" * 80)
print(f"DATA MODE              : {DATA_MODE}")
print(f"Raw train images       : {len(base_train_refined)}")
print(f"Train after augmentation: {len(train_dataset)}")

if val_dataset is not None:
    print(f"Validation images      : {len(val_dataset)}")
else:
    print("Validation images      : NONE")

print(f"Test images            : {len(test_dataset)}")
print(f"Num classes            : {num_classes}")
print("=" * 80)


if DATA_MODE == "dev":
    assert val_dataset is not None
    print("✅ DEVELOPMENT PROTOCOL: 70 / 10 / 20")

elif DATA_MODE == "final":
    assert val_dataset is None
    assert len(base_train_refined) == len(full_train_base)
    print("✅ FINAL PROTOCOL: original 80 / 20")


gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Retrain experiments

In [ ]:
# ================================================================
# FINAL MAIN-TABLE EXPERIMENTS
#
# DATA_MODE phải = "final"
#
# Không tuning nữa.
# Không model selection theo test.
# Model cuối = epoch 15 cố định.
# ================================================================

assert DATA_MODE == "final", (
    "Final experiments bắt buộc DATA_MODE='final'"
)


# ================================================================
# LOCKED TRAINING-OPTIMIZATION RECIPE
# ================================================================

FINAL_OPTIMIZER = "adamw"
FINAL_SCHEDULER = "cosine"

FINAL_LR = 2.5e-4
FINAL_RHO = 0.10

FINAL_LS = 0.1
FINAL_CUTMIX_ALPHA = 1.0

# KD đã khóa
FINAL_KD_ALPHA = 0.7
FINAL_KD_T = 4.0


# ================================================================
# CHỌN NHÓM CẦN CHẠY
# ================================================================

# FINAL_GROUP = "b2"

# FINAL_GROUP = "b0"
FINAL_GROUP = os.environ.get("FINAL_GROUP", "b0")


# ================================================================
# COMMON
# ================================================================

FINAL_BASE = {
    "scheduler_name": FINAL_SCHEDULER,
    "drop_rate": 0.0,
}


# ================================================================
# GROUP 1 — B2
#
# 1. Baseline B2
# 2. B2 + Training Optimization
# 3. B2 + Fusion Teacher
# ================================================================

if FINAL_GROUP == "b2":

    RUN_NAME = "final_b2"

    experiments = [

        # --------------------------------------------------------
        # 1. BASELINE B2
        # --------------------------------------------------------
        {
            **FINAL_BASE,

            "exp_name": "FINAL_B2_Baseline",

            "model_name": "efficientnet_b2",
            "model_variant": "original",

            "optimizer_name": FINAL_OPTIMIZER,

            # Baseline LR:
            # giữ LR chuẩn của baseline
            "lr_value": 5e-4,

            "label_smoothing": 0.0,

            "use_cutmix": False,

            "use_sam": False,
        },


        # --------------------------------------------------------
        # 2. B2 + TRAINING OPTIMIZATION
        # --------------------------------------------------------
        {
            **FINAL_BASE,

            "exp_name": "FINAL_B2_Training_Optimization",

            "model_name": "efficientnet_b2",
            "model_variant": "original",

            "optimizer_name": FINAL_OPTIMIZER,

            "lr_value": FINAL_LR,

            "label_smoothing": FINAL_LS,

            "use_cutmix": True,
            "cutmix_alpha": FINAL_CUTMIX_ALPHA,

            "use_sam": True,
            "sam_rho": FINAL_RHO,
        },


        # --------------------------------------------------------
        # 3. B2 + FUSION = FINAL TEACHER
        # --------------------------------------------------------
        {
            **FINAL_BASE,

            "exp_name": "FINAL_B2_Fusion_Teacher",

            "model_name": "efficientnet_b2",
            "model_variant": "dual_scale",

            "fusion_channels": 512,

            "optimizer_name": FINAL_OPTIMIZER,

            "lr_value": FINAL_LR,

            "label_smoothing": FINAL_LS,

            "use_cutmix": True,
            "cutmix_alpha": FINAL_CUTMIX_ALPHA,

            "use_sam": True,
            "sam_rho": FINAL_RHO,
        },
    ]


# ================================================================
# GROUP 2 — B0
#
# 4. Baseline B0
# 5. B0 + Training Optimization
# ================================================================

elif FINAL_GROUP == "b0":

    RUN_NAME = "final_b0"

    experiments = [

        # --------------------------------------------------------
        # 4. BASELINE B0
        # --------------------------------------------------------
        {
            **FINAL_BASE,

            "exp_name": "FINAL_B0_Baseline",

            "model_name": "efficientnet_b0",
            "model_variant": "original",

            "optimizer_name": FINAL_OPTIMIZER,

            "lr_value": 5e-4,

            "label_smoothing": 0.0,

            "use_cutmix": False,

            "use_sam": False,
        },


        # --------------------------------------------------------
        # 5. B0 + TRAINING OPTIMIZATION
        # --------------------------------------------------------
        {
            **FINAL_BASE,

            "exp_name": "FINAL_B0_Training_Optimization",

            "model_name": "efficientnet_b0",
            "model_variant": "original",

            "optimizer_name": FINAL_OPTIMIZER,

            "lr_value": FINAL_LR,

            "label_smoothing": FINAL_LS,

            "use_cutmix": True,
            "cutmix_alpha": FINAL_CUTMIX_ALPHA,

            "use_sam": True,
            "sam_rho": FINAL_RHO,
        },
    ]


# ================================================================
# GROUP 3 — KD STUDENT
#
# Phải chạy FINAL_B2_Fusion_Teacher trước.
# ================================================================

elif FINAL_GROUP == "kd":

    RUN_NAME = "final_kd"

    # Nếu teacher được train trong CÙNG Kaggle session:
    FINAL_TEACHER_PATH = Path(
        os.environ["KD_TEACHER_PATH"]
    )
    if not FINAL_TEACHER_PATH.exists():
        raise FileNotFoundError(
            "Không tìm thấy FINAL Fusion Teacher checkpoint: "
            f"{FINAL_TEACHER_PATH}"
        )

    experiments = [

        # --------------------------------------------------------
        # 6. KD STUDENT B0
        # --------------------------------------------------------
        {
            **FINAL_BASE,

            "exp_name": "FINAL_B0_KD_Student",

            "model_name": "efficientnet_b0",
            "model_variant": "original",

            "optimizer_name": FINAL_OPTIMIZER,

            "lr_value": FINAL_LR,

            "label_smoothing": FINAL_LS,

            "use_cutmix": True,
            "cutmix_alpha": FINAL_CUTMIX_ALPHA,

            "use_sam": True,
            "sam_rho": FINAL_RHO,

            # KD
            "use_kd": True,

            "kd_teacher_path": str(FINAL_TEACHER_PATH),

            "kd_teacher_variant": "dual_scale",
            "kd_teacher_fusion_channels": 512,

            "kd_alpha": FINAL_KD_ALPHA,
            "kd_temperature": FINAL_KD_T,
        },
    ]


else:
    raise ValueError(
        f"Unknown FINAL_GROUP: {FINAL_GROUP}"
    )


# ================================================================
# PRINT CONFIG
# ================================================================

print("=" * 90)
print("FINAL MAIN-TABLE TRAINING")
print("=" * 90)

print("DATA_MODE       :", DATA_MODE)
print("FINAL_GROUP     :", FINAL_GROUP)
print("RUN_NAME        :", RUN_NAME)
print("EPOCHS          :", epochs)

print("-" * 90)

print("Optimizer       :", FINAL_OPTIMIZER)
print("Scheduler       :", FINAL_SCHEDULER)
print("TO LR           :", FINAL_LR)
print("SAM rho         :", FINAL_RHO)
print("Label smoothing :", FINAL_LS)
print("CutMix alpha    :", FINAL_CUTMIX_ALPHA)

print("-" * 90)

print(
    f"NUMBER OF EXPERIMENTS: "
    f"{len(experiments)}"
)

for i, exp in enumerate(experiments, 1):

    print(
        f"{i:02d}. "
        f"{exp['exp_name']} | "
        f"model={exp.get('model_name')} | "
        f"variant={exp.get('model_variant')} | "
        f"optimizer={exp.get('optimizer_name')} | "
        f"lr={exp.get('lr_value')} | "
        f"LS={exp.get('label_smoothing', 0.0)} | "
        f"CutMix={exp.get('use_cutmix', False)} | "
        f"SAM={exp.get('use_sam', False)} | "
        f"rho={exp.get('sam_rho', '-')}"
    )

print("=" * 90)

In [ ]:
# # ================================================================
# # EXPERIMENT CONFIG
# # Mỗi RUN_NAME tối đa 4 runs (~8h nếu 1 run ~2h)
# # ================================================================

# # RUN_NAME = "opt"
# # RUN_NAME = "to_rho004"
# # RUN_NAME = "to_rho006"
# # RUN_NAME = "to_rho008"
# # RUN_NAME = "to_rho010"
# RUN_NAME = "ablation"

# # Sau khi chạy xong 4 nhóm TO, điền rho thắng theo VAL vào đây.
# SELECTED_RHO = 0.10


# # ================================================================
# # COMMON CONFIG
# # ================================================================

# BASE = {
#     "model_name": "efficientnet_b2",
#     "model_variant": "original",
#     "drop_rate": 0.0,
#     "select_metric": "val_acc",
# }

# OPTIMIZED_BASE = {
#     **BASE,
#     "optimizer_name": "adamw",
#     "scheduler_name": "cosine",
# }


# # ================================================================
# # STAGE 0 — OPTIMIZER / SCHEDULER SELECTION
# # 4 runs, LR cố định 5e-4
# # ================================================================

# if RUN_NAME == "opt":

#     experiments = [
#         {
#             **BASE,
#             "exp_name": "OPT | Adam | None",
#             "optimizer_name": "adam",
#             "scheduler_name": "none",
#             "lr_value": 5e-4,
#             "label_smoothing": 0.0,
#             "use_cutmix": False,
#             "use_sam": False,
#         },
#         {
#             **BASE,
#             "exp_name": "OPT | Adam | Cosine",
#             "optimizer_name": "adam",
#             "scheduler_name": "cosine",
#             "lr_value": 5e-4,
#             "label_smoothing": 0.0,
#             "use_cutmix": False,
#             "use_sam": False,
#         },
#         {
#             **BASE,
#             "exp_name": "OPT | AdamW | None",
#             "optimizer_name": "adamw",
#             "scheduler_name": "none",
#             "lr_value": 5e-4,
#             "label_smoothing": 0.0,
#             "use_cutmix": False,
#             "use_sam": False,
#         },
#         {
#             **BASE,
#             "exp_name": "OPT | AdamW | Cosine",
#             "optimizer_name": "adamw",
#             "scheduler_name": "cosine",
#             "lr_value": 5e-4,
#             "label_smoothing": 0.0,
#             "use_cutmix": False,
#             "use_sam": False,
#         },
#     ]


# # ================================================================
# # STAGE 1 — TABLE 4
# # Joint search: Training Optimization
# #
# # LS=0.1 + CutMix=1.0 + SAM
# # AdamW + cosine
# #
# # 4 rho × 4 LR
# # Chia thành 4 Kaggle sessions, mỗi session 4 runs.
# # ================================================================

# elif RUN_NAME.startswith("to_rho"):

#     rho_map = {
#         "to_rho004": 0.04,
#         "to_rho006": 0.06,
#         "to_rho008": 0.08,
#         "to_rho010": 0.10,
#     }

#     rho = rho_map[RUN_NAME]

#     lr_grid = [
#         (1e-4,   "1e-4"),
#         (2.5e-4, "2.5e-4"),
#         (5e-4,   "5e-4"),
#         (1e-3,   "1e-3"),
#     ]

#     experiments = [
#         {
#             **OPTIMIZED_BASE,
#             "exp_name": f"TO | rho{rho:.2f} | lr{lr_name}",
#             "lr_value": lr_value,
#             "label_smoothing": 0.1,
#             "use_cutmix": True,
#             "cutmix_alpha": 1.0,
#             "use_sam": True,
#             "sam_rho": rho,
#         }
#         for lr_value, lr_name in lr_grid
#     ]


# # ================================================================
# # STAGE 2 — TABLE 5
# # Controlled component ablation
# #
# # LR = 5e-4 cố định.
# # SAM-only sử dụng rho thắng từ Table 4.
# #
# # Baseline KHÔNG chạy lại:
# # dùng kết quả "OPT | AdamW | Cosine".
# #
# # LS+CutMix+SAM cũng KHÔNG chạy lại:
# # lấy cell (SELECTED_RHO, LR=5e-4) từ Table 4.
# #
# # Training Optimization:
# # lấy winner của Table 4.
# # ================================================================

# elif RUN_NAME == "ablation":

#     experiments = [
#         {
#             **OPTIMIZED_BASE,
#             "exp_name": "ABL | LS0.1",
#             "lr_value": 5e-4,
#             "label_smoothing": 0.1,
#             "use_cutmix": False,
#             "use_sam": False,
#         },
#         {
#             **OPTIMIZED_BASE,
#             "exp_name": "ABL | CutMix1.0",
#             "lr_value": 5e-4,
#             "label_smoothing": 0.0,
#             "use_cutmix": True,
#             "cutmix_alpha": 1.0,
#             "use_sam": False,
#         },
#         {
#             **OPTIMIZED_BASE,
#             "exp_name": "ABL | LS0.1 + CutMix1.0",
#             "lr_value": 5e-4,
#             "label_smoothing": 0.1,
#             "use_cutmix": True,
#             "cutmix_alpha": 1.0,
#             "use_sam": False,
#         },
#         {
#             **OPTIMIZED_BASE,
#             "exp_name": f"ABL | SAM only | rho{SELECTED_RHO:.2f}",
#             "lr_value": 5e-4,
#             "label_smoothing": 0.0,
#             "use_cutmix": False,
#             "use_sam": True,
#             "sam_rho": SELECTED_RHO,
#         },
#     ]


# else:
#     raise ValueError(f"Unknown RUN_NAME: {RUN_NAME}")


# print(f"RUN_NAME: {RUN_NAME}")
# print(f"Number of experiments: {len(experiments)}")

# for i, exp in enumerate(experiments, 1):
#     print(f"{i:02d}. {exp['exp_name']}")

In [ ]:

# RUN_NAME = "single_test"
# experiments = [    
#     # {
#     #     "exp_name": "Adam + cosine",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine"
#     # },
#     # {
#     #     "exp_name": "Adam + no scheduler",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "none"
#     # },
#     # {
#     #     "exp_name": "AdamW + no scheduler",
#     #     "optimizer_name": "adamw",
#     #     "scheduler_name": "none"
#     # },
#     # {
#     #     "exp_name": "AdamW + cosine",
#     #     "optimizer_name": "adamw",
#     #     "scheduler_name": "cosine"
#     # },
#     # {
#     #     "exp_name": "Adam + cosine + LS0.1",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1
#     # },
#     # {
#     #     "exp_name": "Adam + cosine + CutMix1.0",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.0,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0
#     # },
#     # {
#     #     "exp_name": "Adam + cosine + SAM0.04",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "use_sam": True,
#     #     "sam_rho": 0.04
#     # },
#     # {
#     #     "exp_name": "Adam + cosine + SAM0.06",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "use_sam": True,
#     #     "sam_rho": 0.06
#     # },
#     # {
#     #     "exp_name": "Adam + cosine + SAM0.07",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "use_sam": True,
#     #     "sam_rho": 0.07
#     # },
#     # {
#     #     "exp_name": "Adam + cosine + SAM0.08",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "use_sam": True,
#     #     "sam_rho": 0.08
#     # },
#     # {
#     #     "exp_name": "Adam + cosine + SAM0.09",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "use_sam": True,
#     #     "sam_rho": 0.09
#     # },
#     # {
#     #     "exp_name": "Adam + cosine + SAM0.11",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "use_sam": True,
#     #     "sam_rho": 0.11
#     # },
#     # {
#     #     "exp_name": "Adam + cosine + SAM0.12",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "use_sam": True,
#     #     "sam_rho": 0.12
#     # },
#     # {
#     #     "exp_name": "Adam + cosine + SAM0.04 + lr1e-4",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 1e-4,
#     #     "use_sam": True,
#     #     "sam_rho": 0.04
#     # },
#     # {
#     #     "exp_name": "Adam + cosine + SAM0.04 + lr2.5e-4",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 2.5e-4,
#     #     "use_sam": True,
#     #     "sam_rho": 0.04
#     # },
#     # {
#     #     "exp_name": "Adam + cosine + SAM0.04 + lr1e-3",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 1e-3,
#     #     "use_sam": True,
#     #     "sam_rho": 0.04
#     # },
#     # {
#     #     "exp_name": "Adam + cosine + Dropout0.2",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": lr,
#     #     "drop_rate": 0.2,
#     #     "use_sam": False
#     # },
#     # {
#     #     "exp_name": "Adam + cosine + Dropout0.3",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": lr,
#     #     "drop_rate": 0.3,
#     #     "use_sam": False
#     # },
#     # {
#     #     "exp_name": "Adam + cosine + Dropout0.4",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": lr,
#     #     "drop_rate": 0.4,
#     #     "use_sam": False
#     # },
#     # {
#     #     "exp_name": "Baseline + LS0.1 + CutMix1.0",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0
#     # },
#     # {
#     #     "exp_name": "Baseline + LS0.1 + CutMi1.0 + SAM0.04 + lr5e-4",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.04,
#     # },
#     # {
#     #     "exp_name": "Baseline + LS0.1 + CutMix1.0 + SAM0.1 + lr2.5e-4",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "lr_value": 2.5e-4,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     # },
#     # {
#     #     "exp_name": "Baseline + LS0.1 + CutMix1.0 + SAM0.04 + lr2.5e-4",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "lr_value": 2.5e-4,
#     #     "use_sam": True,
#     #     "sam_rho": 0.04,
#     # },
#     # {
#     #     "exp_name": "Adamw + LS0.1 + CutMix1.0 + SAM0.04 + lr2.5e-4",
#     #     "optimizer_name": "adamw",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "lr_value": 2.5e-4,
#     #     "use_sam": True,
#     #     "sam_rho": 0.04,
#     # },
#     # {
#     #     "exp_name": "Adamw + LS0.1 + CutMix1.0 + SAM0.1 + lr2.5e-4",
#     #     "optimizer_name": "adamw",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "lr_value": 2.5e-4,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     # },
#     # {
#     #     "exp_name": "B0 baseline",
#     #     "model_name": "efficientnet_b0",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 5e-4,
#     #     "label_smoothing": 0.0,
#     #     "use_cutmix": False,
#     #     "use_sam": False,
#     #     "drop_rate": 0.0
#     # },
#     # {
#     #     "exp_name": "B0 + SAM0.1",
#     #     "model_name": "efficientnet_b0",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 5e-4,
#     #     "label_smoothing": 0.0,
#     #     "use_cutmix": False,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "drop_rate": 0.0
#     # },
#     # {
#     #     "exp_name": "B0 + SAM0.1 + lr2.5e-4",
#     #     "model_name": "efficientnet_b0",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 2.5e-4,
#     #     "label_smoothing": 0.0,
#     #     "use_cutmix": False,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1
#     # },
#     # {
#     #     "exp_name": "Strong B2 + GeM",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "gem",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "lr_value": 2.5e-4,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1
#     # },
#     # {
#     #     "exp_name": "Strong B2 + late ECA",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "late_eca",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "lr_value": 2.5e-4,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1
#     # },
#     # {
#     #     "exp_name": "Strong B2 + Dual Scale",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "dual_scale",
#     #     "fusion_channels": 512,
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "lr_value": 2.5e-4,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1
#     # },
#     # {
#     #     "exp_name": "Strong B2 + Dual Scale + ECA",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "dual_scale_eca",
#     #     "fusion_channels": 512,
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "lr_value": 2.5e-4,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,        
#     # },
#     # {
#     #     "exp_name": "Strong B2 + Dual Scale + ECA + GeM",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "dual_scale_eca_gem",
#     #     "fusion_channels": 512,
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "lr_value": 2.5e-4,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     # },
#     # {
#     #     "exp_name": "Strong B2 + Dual Scale + GeM",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "dual_scale_gem",
#     #     "fusion_channels": 512,
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "lr_value": 2.5e-4,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     # },
#     # {
#     #     "exp_name": "Strong B2 + Dual Scale + ArcFace",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "dual_scale_arcface",
#     #     "fusion_channels": 512,
#     #     "embedding_dim": 256,
#     #     "arcface_s": 30.0,
#     #     "arcface_m": 0.3,
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "lr_value": 2.5e-4,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     # },
#     # {
#     #     "exp_name": "Strong B2 + Dual Scale + Gated Fusion",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "dual_scale_gated",
#     #     "fusion_channels": 512,
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "lr_value": 2.5e-4,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1
#     # },
#     # {
#     #     "exp_name": "Strong B2 + Dual Scale + Aux C4",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "dual_scale_aux",
#     #     "fusion_channels": 512,
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "lr_value": 2.5e-4,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "aux_weight": 0.2
#     # },
#     # {
#     #     "exp_name": "HAM10000 Plain Baseline",
#     #     "model_name": "efficientnet_b2",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.0,
#     #     "use_cutmix": False,
#     #     "cutmix_alpha": 0.0,
#     #     "lr_value": 5e-4,
#     #     "use_sam": False,
#     #     "sam_rho": 0.0
#     # },
#     # {
#     #     "exp_name": "HAM10000 Plain Baseline + Dual Scale",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "dual_scale",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.0,
#     #     "use_cutmix": False,
#     #     "cutmix_alpha": 0.0,
#     #     "lr_value": 5e-4,
#     #     "use_sam": False,
#     #     "sam_rho": 0.0
#     # },
#     # {
#     #     "exp_name": "HAM10000 Strong Baseline",
#     #     "model_name": "efficientnet_b2",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "lr_value": 2.5e-4,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1
#     # },
#     # {
#     #     "exp_name": "HAM10000 Strong Baseline + Dual Scale",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "dual_scale",
#     #     "fusion_channels": 512,
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "lr_value": 2.5e-4,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1
#     # },
#     # {
#     #     "exp_name": "HAM10000 Plain Baseline + Dual Scale ArcFace clean",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "dual_scale_arcface",
#     #     "fusion_channels": 512,
#     #     "embedding_dim": 256,
#     #     "arcface_s": 24.0,
#     #     "arcface_m": 0.15,
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 5e-4,
#     # },
#     # {
#     #     "exp_name": "HAM10000 Plain B2 + ArcFace",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "arcface",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "embedding_dim": 256,
#     #     "arcface_s": 24.0,
#     #     "arcface_m": 0.15,
#     #     "lr_value": 5e-4,
#     # },
#     # {
#     #     "exp_name": "Skin31 Plain B2 + Regional Head",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "regional",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "embedding_dim": 256,
#     #     "region_grid": 2,
#     #     "use_gem": True,
#     #     "attn_hidden": 128,
#     #     "lr_value": 5e-4
#     # },
#     # {
#     #     "exp_name": "Skin31 Strong B2 + Regional Head",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "regional",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "embedding_dim": 256,
#     #     "region_grid": 2,
#     #     "use_gem": True,
#     #     "attn_hidden": 128,
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "lr_value": 2.5e-4,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1
#     # },
#     # {
#     #     "exp_name": "Skin31 Plain B2 + Spatial Refine",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "spatial_refine",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 5e-4,
#     #     "refine_kernel_size": 7
#     # },
#     # {
#     #     "exp_name": "Skin31 Plain B2 + Compact Head",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "compact_head",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 5e-4,
#     #     "embedding_dim": 256,
#     #     "use_gem": True,
#     #     "drop_rate": 0.2,
#     #     "use_cosine_classifier": False,
#     #     "cosine_s": 24.0
#     # },
#     # {
#     #     "exp_name": "Skin31 Plain B2 + Compact Head + Cosine",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "compact_head",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 5e-4,
#     #     "embedding_dim": 256,
#     #     "use_gem": True,
#     #     "drop_rate": 0.2,
#     #     "use_cosine_classifier": True,
#     #     "cosine_s": 24.0
#     # },
#     # {
#     #     "exp_name": "Skin31 Plain B2 + Dual Evidence",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "dual_evidence",
#     #     "lr_value": 5e-4,
#     #     "drop_rate": 0.0,
#     #     "embedding_dim": 256,
#     #     "region_grid": 3,
#     #     "top_k": 3,
#     #     "use_gem": True,
#     #     "score_hidden": 128
#     # },
#     # {
#     #     "exp_name": "Skin31 Plain B2 + Dual Evidence k1",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "dual_evidence",
#     #     "lr_value": 5e-4,
#     #     "drop_rate": 0.0,
#     #     "embedding_dim": 256,
#     #     "region_grid": 3,
#     #     "top_k": 1,
#     #     "use_gem": True,
#     #     "score_hidden": 128
#     # },
#     # {
#     #     "exp_name": "Skin31 Plain B2 + Dual Evidence k5",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "dual_evidence",
#     #     "lr_value": 5e-4,
#     #     "drop_rate": 0.0,
#     #     "embedding_dim": 256,
#     #     "region_grid": 3,
#     #     "top_k": 5,
#     #     "use_gem": True,
#     #     "score_hidden": 128
#     # },
#     # {
#     #     "exp_name": "Skin31 Strong B2 + C4 Refine",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "c4_refine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "lr_value": 2.5e-4,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1
#     # },
#     # {
#     #     "exp_name": "EfficientNet-B2 Strong + Mamba Refine d96",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 2.5e-4,
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "mamba_refine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "mamba_dim": 96,
#     #     "d_state": 16,
#     #     "d_conv": 3,
#     #     "expand": 2,
#     #     "init_alpha": 0.1,
#     #     "use_external_mamba": False
#     # },
#     # {
#     #     "exp_name": "EfficientNet-B2 Strong + Mamba Refine d64 a0.05",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 2.5e-4,
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "mamba_refine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "mamba_dim": 64,
#     #     "d_state": 8,
#     #     "d_conv": 3,
#     #     "expand": 1,
#     #     "init_alpha": 0.05,
#     #     "use_external_mamba": False
#     # },
#     # {
#     #     "exp_name": "EfficientNet-B2 Strong + UPG Head",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 2.5e-4,
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "upg",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "embedding_dim": 256,
#     #     "prototype_scale": 12.0,
#     #     "classifier_scale": 1.0,
#     #     "uncertainty_hidden": 128,
#     #     "use_gem": True
#     # },
#     # {
#     #     "exp_name": "EfficientNet-B2 Strong + UPG Scalar ps16",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 2.5e-4,
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "upg",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "drop_rate": 0.0,
#     #     "embedding_dim": 256,
#     #     "prototype_scale": 16.0,
#     #     "classifier_scale": 1.0,
#     #     "uncertainty_hidden": 128,
#     #     "use_gem": True
#     # },
#     # {
#     #     "exp_name": "EfficientNet-B2 Strong + UPG Scalar e384 ps12",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 2.5e-4,
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "upg",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "drop_rate": 0.0,
#     #     "embedding_dim": 384,
#     #     "prototype_scale": 12.0,
#     #     "classifier_scale": 1.0,
#     #     "uncertainty_hidden": 128,
#     #     "use_gem": True
#     # },
#     # {
#     #     "exp_name": "EfficientNet-B2 Strong + UPG Scalar e512 ps12",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 2.5e-4,
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "upg",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "drop_rate": 0.0,
#     #     "embedding_dim": 512,
#     #     "prototype_scale": 12.0,
#     #     "classifier_scale": 1.0,
#     #     "uncertainty_hidden": 128,
#     #     "use_gem": True
#     # },
#     # {
#     #     "exp_name": "EfficientNet-B2 Strong + UPG Scalar e384 uh256 ps12",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 2.5e-4,
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "upg",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "drop_rate": 0.0,
#     #     "embedding_dim": 384,
#     #     "prototype_scale": 12.0,
#     #     "classifier_scale": 1.0,
#     #     "uncertainty_hidden": 256,
#     #     "use_gem": True
#     # },
#     #     {
#     #     "exp_name": "EfficientNet-B2 Strong + UPG Scalar e384 ps10",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 2.5e-4,
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "upg",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "drop_rate": 0.0,
#     #     "embedding_dim": 384,
#     #     "prototype_scale": 10.0,
#     #     "classifier_scale": 1.0,
#     #     "uncertainty_hidden": 128,
#     #     "use_gem": True
#     # },
#     # {
#     #     "exp_name": "EfficientNet-B2 Strong + UPG Scalar e384 ps12 cs0.8",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 2.5e-4,
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "upg",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "drop_rate": 0.0,
#     #     "embedding_dim": 384,
#     #     "prototype_scale": 12.0,
#     #     "classifier_scale": 0.8,
#     #     "uncertainty_hidden": 128,
#     #     "use_gem": True
#     # },
#     # {
#     #     "exp_name": "EfficientNet-B2 Strong + UPG e384 ps12 SelfDistill a0.3 T4",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 2.5e-4,
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "upg",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "drop_rate": 0.0,
#     #     "embedding_dim": 384,
#     #     "prototype_scale": 12.0,
#     #     "classifier_scale": 1.0,
#     #     "uncertainty_hidden": 128,
#     #     "use_gem": True,
#     #     "use_kd": True,
#     #     "kd_teacher_path": "/kaggle/input/models/thanhkhil/efficientnet-b2-skin31/pytorch/best_upg/1/efficientnet-b2_strong_plus_upg_scalar_e384_ps12_best.pth",
#     #     "kd_alpha": 0.3,
#     #     "kd_temperature": 4.0
#     # },
#     # {
#     #     "exp_name": "EfficientNet-B2 Strong + UPG e384 ps12 SelfDistill a0.5 T4",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 2.5e-4,
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "upg",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "drop_rate": 0.0,
#     #     "embedding_dim": 384,
#     #     "prototype_scale": 12.0,
#     #     "classifier_scale": 1.0,
#     #     "uncertainty_hidden": 128,
#     #     "use_gem": True,
#     #     "use_kd": True,
#     #     "kd_teacher_path": "/kaggle/input/models/thanhkhil/efficientnet-b2-skin31/pytorch/best_upg/1/efficientnet-b2_strong_plus_upg_scalar_e384_ps12_best.pth",
#     #     "kd_alpha": 0.5,
#     #     "kd_temperature": 4.0
#     # },
#     # {
#     #     "exp_name": "EfficientNet-B2 Strong + HierProto-UPG e384 ps12 cw0.05",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 2.5e-4,
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "upg",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "drop_rate": 0.0,
#     #     "embedding_dim": 384,
#     #     "prototype_scale": 12.0,
#     #     "classifier_scale": 1.0,
#     #     "uncertainty_hidden": 128,
#     #     "use_gem": True,
#     #     "num_coarse_classes": num_coarse_classes,
#     #     "fine_to_coarse": fine_to_coarse_list,
#     #     "residual_scale": 0.5,
#     #     "coarse_scale": 12.0,
#     #     "coarse_weight": 0.05,
#     #     "compact_weight": 0.0,
#     #     "sep_weight": 0.0
#     # },
#     # {
#     #     "exp_name": "EfficientNet-B2 Strong + HierProto-UPG e384 ps12 cw0.1",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 2.5e-4,
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "upg",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "drop_rate": 0.0,
#     #     "embedding_dim": 384,
#     #     "prototype_scale": 12.0,
#     #     "classifier_scale": 1.0,
#     #     "uncertainty_hidden": 128,
#     #     "use_gem": True,
#     #     "num_coarse_classes": num_coarse_classes,
#     #     "fine_to_coarse": fine_to_coarse_list,
#     #     "residual_scale": 0.5,
#     #     "coarse_scale": 12.0,
#     #     "coarse_weight": 0.1,
#     #     "compact_weight": 0.0,
#     #     "sep_weight": 0.0
#     # },
#     # {
#     #     "exp_name": "EfficientNet-B2 Strong + HierProto-UPG e384 ps12 cw0.2",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 2.5e-4,
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "upg",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "drop_rate": 0.0,
#     #     "embedding_dim": 384,
#     #     "prototype_scale": 12.0,
#     #     "classifier_scale": 1.0,
#     #     "uncertainty_hidden": 128,
#     #     "use_gem": True,
#     #     "num_coarse_classes": num_coarse_classes,
#     #     "fine_to_coarse": fine_to_coarse_list,
#     #     "residual_scale": 0.5,
#     #     "coarse_scale": 12.0,
#     #     "coarse_weight": 0.2,
#     #     "compact_weight": 0.0,
#     #     "sep_weight": 0.0
#     # },
#     # {
#     #     "exp_name": "EfficientNet-B2 Strong + HierProto-UPG e384 ps12 cw0.2 rs1.0",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 2.5e-4,
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "upg",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "drop_rate": 0.0,
#     #     "embedding_dim": 384,
#     #     "prototype_scale": 12.0,
#     #     "classifier_scale": 1.0,
#     #     "uncertainty_hidden": 128,
#     #     "use_gem": True,
#     #     "num_coarse_classes": num_coarse_classes,
#     #     "fine_to_coarse": fine_to_coarse_list,
#     #     "residual_scale": 1.0,
#     #     "coarse_scale": 12.0,
#     #     "coarse_weight": 0.2,
#     #     "compact_weight": 0.0,
#     #     "sep_weight": 0.0
#     # },
#     # {
#     #     "exp_name": "EfficientNet-B2 Strong + HierProto-UPG e384 ps10 cw0.2 rs0.5",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 2.5e-4,
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "upg",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "drop_rate": 0.0,
#     #     "embedding_dim": 384,
#     #     "prototype_scale": 10.0,
#     #     "classifier_scale": 1.0,
#     #     "uncertainty_hidden": 128,
#     #     "use_gem": True,
#     #     "num_coarse_classes": num_coarse_classes,
#     #     "fine_to_coarse": fine_to_coarse_list,
#     #     "residual_scale": 0.5,
#     #     "coarse_scale": 12.0,
#     #     "coarse_weight": 0.2,
#     #     "compact_weight": 0.0,
#     #     "sep_weight": 0.0
#     # },
#     # {
#     #     "exp_name": "EfficientNet-B2 Strong + HierProto-UPG e384 ps12 cw0.2 rs1.0 gateT0.5",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "lr_value": 2.5e-4,
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "upg",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "drop_rate": 0.0,
#     #     "embedding_dim": 384,
#     #     "prototype_scale": 12.0,
#     #     "classifier_scale": 1.0,
#     #     "uncertainty_hidden": 128,
#     #     "use_gem": True,
#     #     "num_coarse_classes": num_coarse_classes,
#     #     "fine_to_coarse": fine_to_coarse_list,
#     #     "residual_scale": 1.0,
#     #     "coarse_scale": 12.0,
#     #     "coarse_weight": 0.2,
#     #     "compact_weight": 0.0,
#     #     "sep_weight": 0.0,
#     #     "gate_temperature": 0.5,
#     #     "run_diagnostic": True,
#     #     "baseline_checkpoint": "/kaggle/input/models/thanhkhil/efficientnet-b2-skin31/pytorch/baseline-ls-cutmix-sam/1/baseline_plus_ls0.1_plus_cutmix1.0_plus_sam0.1_plus_lr2.5e-4_best.pth"
#     # },
#     # {
#     #     "exp_name": "WAGF EfficientNet-B2 + LS0.1 + CutMix1.0 + SAM0.1",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "wagf",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "lr_value": 2.5e-4,
#     #     "drop_rate": 0.0,
#     #     "run_diagnostic": False
#     # },
#     # {
#     #     "exp_name": "ECA EfficientNet-B2 + LS0.1 + CutMix1.0 + SAM0.1",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "eca",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "lr_value": 2.5e-4,
#     #     "drop_rate": 0.0
#     # },
#     # {
#     #     "exp_name": "CBAM EfficientNet-B2 + LS0.1 + CutMix1.0 + SAM0.1",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "cbam",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "lr_value": 2.5e-4,
#     #     "drop_rate": 0.0
#     # },
#     # {
#     #     "exp_name": "MSSF EfficientNet-B2 + LS0.1 + CutMix1.0 + SAM0.1",
#     #     "model_name": "efficientnet_b2",
#     #     "model_variant": "mssf",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "lr_value": 2.5e-4,
#     #     "drop_rate": 0.0
#     # },
#     # {
#     #     "exp_name": "B0 + LS0.1 + CutMix1.0 + SAM0.1",
#     #     "model_name": "efficientnet_b0",
#     #     "optimizer_name": "adam",
#     #     "scheduler_name": "cosine",
#     #     "label_smoothing": 0.1,
#     #     "use_cutmix": True,
#     #     "cutmix_alpha": 1.0,
#     #     "use_sam": True,
#     #     "sam_rho": 0.1,
#     #     "lr_value": 2.5e-4,
#     #     "drop_rate": 0.0,
#     # }
# ]

In [ ]:
import inspect
import gc

RUN_EXP_PARAMS = set(inspect.signature(run_experiment).parameters.keys())


def make_failed_result(exp, err):
    return {
        "experiment": exp["exp_name"],
        "data_mode": DATA_MODE,

        "model_name": exp.get("model_name", "efficientnet_b2"),
        "model_variant": exp.get("model_variant", "original"),

        "optimizer": exp.get("optimizer_name"),
        "scheduler": exp.get("scheduler_name"),

        "lr": float(
            exp["lr_value"]
            if exp.get("lr_value") is not None
            else lr
        ),

        "label_smoothing": float(
            exp.get("label_smoothing", 0.0)
        ),

        "use_cutmix": bool(
            exp.get("use_cutmix", False)
        ),

        "cutmix_alpha": float(
            exp.get("cutmix_alpha", 1.0)
        ),

        "use_sam": bool(
            exp.get("use_sam", False)
        ),

        "sam_rho": float(
            exp.get("sam_rho", 0.05)
        ),

        "drop_rate": float(
            exp.get("drop_rate", 0.0)
        ),

        # DEV
        "best_epoch": None,
        "best_val_score": None,
        "best_val_acc": None,
        "best_val_balanced_acc": None,
        "best_val_macro_f1": None,
        "best_val_weighted_f1": None,
        "best_val_loss": None,

        # FINAL
        "final_epoch": None,

        # TEST
        "test_acc": None,
        "test_balanced_acc": None,
        "test_macro_f1": None,
        "test_macro_precision": None,
        "test_macro_recall": None,
        "test_weighted_f1": None,
        "test_loss": None,

        "model_selection": None,

        "best_acc": None,
        "save_path": None,

        "status": f"failed: {repr(err)}",
    }


# ================================================================
# RUN
# ================================================================

print("=" * 100)
print(f"RUN NAME: {RUN_NAME}")
print(f"NUMBER OF EXPERIMENTS: {len(experiments)}")
print("=" * 100)

for i, exp in enumerate(experiments, 1):

    optimizer_name = exp.get("optimizer_name", "N/A")
    scheduler_name = exp.get("scheduler_name", "N/A")
    lr_value = exp.get("lr_value", lr)

    print(
        f"{i:02d}/{len(experiments)}  "
        f"{exp['exp_name']} | "
        f"optimizer={optimizer_name} | "
        f"scheduler={scheduler_name} | "
        f"lr={lr_value}"
    )

print("=" * 100)


all_results = []

for i, exp in enumerate(experiments, 1):

    optimizer_name = exp.get("optimizer_name", "N/A")
    scheduler_name = exp.get("scheduler_name", "N/A")
    lr_value = exp.get("lr_value", lr)

    print("\n" + "#" * 100)
    print(f"EXPERIMENT {i}/{len(experiments)}")
    print(f"Name      : {exp['exp_name']}")
    print(f"Optimizer : {optimizer_name}")
    print(f"Scheduler : {scheduler_name}")
    print(f"LR        : {lr_value}")
    print(
        f"LS        : {exp.get('label_smoothing', 0.0)} | "
        f"CutMix    : {exp.get('use_cutmix', False)} "
        f"(alpha={exp.get('cutmix_alpha', 1.0)}) | "
        f"SAM       : {exp.get('use_sam', False)} "
        f"(rho={exp.get('sam_rho', 0.05)})"
    )
    print("#" * 100)

    unknown = [
        k for k in exp.keys()
        if k not in RUN_EXP_PARAMS
    ]

    if unknown:
        print(
            f"[WARN] {exp['exp_name']}: "
            f"bỏ qua key không hợp lệ -> {unknown}"
        )

    kwargs = {
        k: v
        for k, v in exp.items()
        if k in RUN_EXP_PARAMS
    }

    try:
        result = run_experiment(**kwargs)

    except Exception as e:
        result = make_failed_result(exp, e)

        print(
            f"[{exp['exp_name']}] FAILED -> "
            f"{repr(e)}"
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    all_results.append(result)

    summary_df, summary_csv_path, summary_json_path = \
        save_summary_files(all_results)


# ================================================================
# SUMMARY
# selection / ranking = VALIDATION ONLY
# ================================================================

summary_df = pd.DataFrame(all_results)

if DATA_MODE == "dev":

    if "best_val_acc" in summary_df.columns:
        summary_df = summary_df.sort_values(
            by="best_val_acc",
            ascending=False,
            na_position="last",
        ).reset_index(drop=True)

    summary_title = (
        "FINAL SUMMARY — SORTED BY VALIDATION ACCURACY"
    )

else:

    # FINAL: KHÔNG rank theo test
    summary_df = summary_df.reset_index(drop=True)

    summary_title = (
        "FINAL SUMMARY — FIXED-EPOCH FINAL EVALUATION"
    )


summary_df = order_summary_columns(summary_df)

summary_df.to_csv(
    work_dir / f"{RUN_NAME}_summary.csv",
    index=False
)

print("\n" + "=" * 100)
print(summary_title)
print("=" * 100)

main_cols = [
    c for c in [
        "experiment",
        "data_mode",
        "model_name",
        "model_variant",
        "optimizer",
        "scheduler",
        "lr",
        "label_smoothing",
        "use_cutmix",
        "use_sam",
        "sam_rho",

        "best_epoch",
        "final_epoch",

        "best_val_acc",

        "test_acc",
        "test_balanced_acc",
        "test_macro_precision",
        "test_macro_recall",
        "test_macro_f1",
        "test_weighted_f1",
        "test_loss",

        "model_selection",
        "status",
    ]
    if c in summary_df.columns
]

with pd.option_context(
    "display.float_format", "{:.4f}".format,
    "display.max_columns", None,
    "display.width", 220,
):
    display(summary_df[main_cols])


# ================================================================
# LEARNING CURVES
# ================================================================

all_history_list = []

# Chỉ lấy experiment THỰC SỰ success trong batch hiện tại
successful_exp_names = {
    r["experiment"]
    for r in all_results
    if r.get("status") == "ok"
}


for exp in experiments:

    if exp["exp_name"] not in successful_exp_names:
        continue

    csv_path = (
        history_dir /
        f"{slugify_exp_name(exp['exp_name'])}_history.csv"
    )

    if csv_path.exists():

        df_hist = pd.read_csv(csv_path)

        # Double-check đúng experiment
        if (
            "experiment" in df_hist.columns
            and len(df_hist) > 0
            and df_hist["experiment"].iloc[0] == exp["exp_name"]
        ):
            all_history_list.append(df_hist)


if len(all_history_list) > 0:

    all_history = pd.concat(
        all_history_list,
        ignore_index=True
    )

    all_history.to_csv(
        work_dir / f"{RUN_NAME}_history.csv",
        index=False
    )

    if DATA_MODE == "dev":

        curve_cols = [
            c for c in [
                "train_acc",
                "train_loss",
                "val_acc",
                "val_loss",
                "val_macro_f1",
                "val_balanced_acc",
            ]
            if c in all_history.columns
        ]
    
    else:
    
        curve_cols = [
            c for c in [
                "train_acc",
                "train_loss",
                "test_acc",
                "test_loss",
                "test_macro_f1",
                "test_balanced_acc",
            ]
            if c in all_history.columns
        ]

    display(
        all_history[
            ["experiment", "epoch"] + curve_cols
        ]
    )

    if DATA_MODE == "dev":

        plot_specs = [
            ("train_acc", "Train Accuracy"),
            ("val_acc", "Validation Accuracy"),
            ("val_macro_f1", "Validation Macro F1"),
            ("val_balanced_acc", "Validation Balanced Accuracy"),
            ("train_loss", "Train Loss"),
            ("val_loss", "Validation Loss"),
        ]
    
    else:
    
        plot_specs = [
            ("train_acc", "Train Accuracy"),
            ("test_acc", "Test Accuracy"),
            ("test_macro_f1", "Test Macro F1"),
            ("test_balanced_acc", "Test Balanced Accuracy"),
            ("train_loss", "Train Loss"),
            ("test_loss", "Test Loss"),
        ]
    
    
    for col, title in plot_specs:
    
        if col not in all_history.columns:
            continue
    
        plt.figure(figsize=(12, 5))
    
        for exp_name in all_history["experiment"].unique():
    
            df_exp = all_history[
                all_history["experiment"] == exp_name
            ]
    
            plt.plot(
                df_exp["epoch"],
                df_exp[col],
                marker="o",
                ms=3,
                label=exp_name,
            )
    
        plt.xlabel("Epoch")
        plt.ylabel(col)
        plt.title(f"{RUN_NAME} — {title}")
        plt.grid(alpha=0.3)
        plt.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

else:
    print("No history files found.")

In [ ]:
# TEACHER_PATH = "/kaggle/input/models/thanhkhil/efficientnet-b2-skin31/pytorch/best-dualscale/1/strong_b2_plus_dual_scale_best.pth"
 
# kd_result = run_experiment(
#     exp_name="B0 + KD(DualScale) + LS0.1 + CutMix1.0 + SAM0.1",
#     model_name="efficientnet_b0",         # STUDENT
#     model_variant="original",
#     optimizer_name="adam",
#     scheduler_name="cosine",
#     label_smoothing=0.1,
#     use_cutmix=True,
#     cutmix_alpha=1.0,
#     use_sam=True,
#     sam_rho=0.1,
#     lr_value=2.5e-4,
#     drop_rate=0.0,
#     # ── KD ──────────────────────────────────────────
#     use_kd=True,
#     kd_teacher_path=TEACHER_PATH,
#     kd_teacher_variant="dual_scale",       # teacher kiến trúc
#     kd_teacher_fusion_channels=512,        # SỬA nếu teacher train với fc khác (256?)
#     kd_alpha=0.5,                          # 0.5 CE + 0.5 KD (gap nhỏ → cân bằng)
#     kd_temperature=4.0,
#     run_diagnostic=False,
# )
# kd_result_07 = run_experiment(
#     exp_name="B0 + KD(DualScale) a0.7 + LS0.1 + CutMix1.0 + SAM0.1",
#     model_name="efficientnet_b0",
#     model_variant="original",
#     optimizer_name="adam",
#     scheduler_name="cosine",
#     label_smoothing=0.1,
#     use_cutmix=True,
#     cutmix_alpha=1.0,
#     use_sam=True,
#     sam_rho=0.1,
#     lr_value=2.5e-4,
#     drop_rate=0.0,
#     use_kd=True,
#     kd_teacher_path=TEACHER_PATH,
#     kd_teacher_variant="dual_scale",
#     kd_teacher_fusion_channels=512,
#     kd_alpha=0.7,                          # ← đổi
#     kd_temperature=4.0,
#     run_diagnostic=False,
# )

# kd_result_03 = run_experiment(
#     exp_name="B0 + KD(DualScale) a0.3 + LS0.1 + CutMix1.0 + SAM0.1",
#     model_name="efficientnet_b0",
#     model_variant="original",
#     optimizer_name="adam",
#     scheduler_name="cosine",
#     label_smoothing=0.1,
#     use_cutmix=True,
#     cutmix_alpha=1.0,
#     use_sam=True,
#     sam_rho=0.1,
#     lr_value=2.5e-4,
#     drop_rate=0.0,
#     use_kd=True,
#     kd_teacher_path=TEACHER_PATH,
#     kd_teacher_variant="dual_scale",
#     kd_teacher_fusion_channels=512,
#     kd_alpha=0.3,                          # ← đổi
#     kd_temperature=4.0,
#     run_diagnostic=False,
# )

# kd_result_06 = run_experiment(
#     exp_name="B0 + KD(DualScale) a0.6 + LS0.1 + CutMix1.0 + SAM0.1",
#     model_name="efficientnet_b0",
#     model_variant="original",
#     optimizer_name="adam",
#     scheduler_name="cosine",
#     label_smoothing=0.1,
#     use_cutmix=True,
#     cutmix_alpha=1.0,
#     use_sam=True,
#     sam_rho=0.1,
#     lr_value=2.5e-4,
#     drop_rate=0.0,
#     use_kd=True,
#     kd_teacher_path=TEACHER_PATH,
#     kd_teacher_variant="dual_scale",
#     kd_teacher_fusion_channels=512,
#     kd_alpha=0.6,                          # ← đổi
#     kd_temperature=4.0,
#     run_diagnostic=False,
# )

# kd_result_08 = run_experiment(
#     exp_name="B0 + KD(DualScale) a0.8 + LS0.1 + CutMix1.0 + SAM0.1",
#     model_name="efficientnet_b0",
#     model_variant="original",
#     optimizer_name="adam",
#     scheduler_name="cosine",
#     label_smoothing=0.1,
#     use_cutmix=True,
#     cutmix_alpha=1.0,
#     use_sam=True,
#     sam_rho=0.1,
#     lr_value=2.5e-4,
#     drop_rate=0.0,
#     use_kd=True,
#     kd_teacher_path=TEACHER_PATH,
#     kd_teacher_variant="dual_scale",
#     kd_teacher_fusion_channels=512,
#     kd_alpha=0.8,                          # ← đổi
#     kd_temperature=4.0,
#     run_diagnostic=False,
# )